# Tourist RAG: поиск и ассистент по достопримечательностям

Этот ноутбук превращает базовый **RAG по туристическим данным** в проект, максимально похожий на реальные задачи команды ассистентского поиска по товарам/объявлениям:

- строим **векторный поиск** (dense retrieval) + **лексический** (BM25) и **гибрид**;
- делаем **pipeline валидации**: retrieval-метрики (Recall@k, MRR, nDCG) + простые LLM-метрики (Answer Relevancy);
- фиксируем, *почему* выбран такой подход и какие есть альтернативы.

> **Важно:** внешние LLM API (OpenAI и т.п.) НЕ используются. Генерация — локально через модели HuggingFace (можно запускать в Colab/локально).


## 0) Установка зависимостей




In [56]:
# ====== Colab: one-time installs (RUN ONCE) ======
!pip -q install -U \
  "pandas>=2.0" "numpy>=1.24" "tqdm" "scikit-learn" "matplotlib" "umap-learn" \
  "sentence-transformers>=2.6" "faiss-cpu" "rank-bm25" \
  "transformers>=4.44.0" "accelerate>=0.33.0" "bitsandbytes>=0.43.0" \
  "safetensors>=0.4.3" "huggingface_hub>=0.24.0" "einops" \
  "gdown"

import pandas, numpy, sklearn, matplotlib
import sentence_transformers, faiss
import transformers, accelerate, bitsandbytes, huggingface_hub

print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("sklearn:", sklearn.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("faiss:", faiss.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

print("\nIMPORTANT: Runtime -> Restart runtime (обязательно).")

pandas: 3.0.1
numpy: 2.0.2
sklearn: 1.8.0
sentence-transformers: 5.2.3
faiss: 1.13.2
transformers: 5.3.0
accelerate: 1.13.0
bitsandbytes: 0.49.2
huggingface_hub: 1.6.0

IMPORTANT: Runtime -> Restart runtime (обязательно).


## 1) Импорты и настройки

Здесь:
- задаём seed для воспроизводимости;
- описываем единый `CONFIG`

In [57]:
import os
import re
import math
import json
import random
from dataclasses import dataclass
from typing import List, Dict, Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

@dataclass
class Config:
    # Данные
    gdrive_url: str = 'https://drive.google.com/uc?id=1P1BsvI2jPN3fEqjc2YZxmQ-MTs22WVUk'
    data_path: str = "data.csv"

    # Чанкинг
    chunk_size_chars: int = 900      # грубо ~150-250 токенов (зависит от языка)
    chunk_overlap_chars: int = 150

    # Retrieval
    top_k_dense: int = 20
    top_k_bm25: int = 20
    top_k_hybrid: int = 8            # финальное k в контексте для генератора
    bm25_k1: float = 1.5
    bm25_b: float = 0.75
    hybrid_alpha: float = 0.5        # 0 = только BM25, 1 = только dense

    # Модели
    embedding_model = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    rerank_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"  # опционально
    use_rerank: bool = False          # включить reranking (если модель доступна)
    rerank_top_k: int = 30            # сколько кандидатов rerank-ать (из retrieval)

    gen_model: str = "google/flan-t5-base"  # компактная, хорошо для QA/инструкций

    # Оценка
    eval_queries: int = 80
    retrieval_k: int = 10
    ar_num_questions: int = 3        # K в Answer Relevancy

    # Faithfulness (насколько ответ опирается на контекст)
    faith_min_overlap: float = 0.18  # порог перекрытия токенов 'утверждение ↔ контекст'
    faith_max_claims: int = 8        # сколько предложений-утверждений проверяем


CONFIG = Config()

print(CONFIG)


Config(gdrive_url='https://drive.google.com/uc?id=1P1BsvI2jPN3fEqjc2YZxmQ-MTs22WVUk', data_path='data.csv', chunk_size_chars=900, chunk_overlap_chars=150, top_k_dense=20, top_k_bm25=20, top_k_hybrid=8, bm25_k1=1.5, bm25_b=0.75, hybrid_alpha=0.5, rerank_model='cross-encoder/ms-marco-MiniLM-L-6-v2', use_rerank=False, rerank_top_k=30, gen_model='google/flan-t5-base', eval_queries=80, retrieval_k=10, ar_num_questions=3, faith_min_overlap=0.18, faith_max_claims=8)


## 2) Загрузка данных

Скачиваем CSV с Google Drive и читаем в `pandas`.


In [58]:
from pathlib import Path
import gdown

url = "https://drive.google.com/uc?id=1P1BsvI2jPN3fEqjc2YZxmQ-MTs22WVUk"
path = Path("data.csv")

if not path.exists():
    try:
        gdown.download(url, str(path), quiet=False)
    except Exception as e:
        raise RuntimeError(
            "Не удалось скачать через gdown (Drive может требовать подтверждение/лимит). "
            "Скачай data.csv вручную по ссылке и положи рядом с ноутбуком."
        ) from e

In [59]:
data = pd.read_csv("data.csv")
print(data.shape)
data.head(3)

(14634, 9)


,Unnamed: 0,Name,WikiData,City,Lon,Lat,description,image,en_txt
0,0,Динамо,Q37996725,Екатеринбург,60.600349,56.845398,спорткомплекс в Екатеринбурге,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUEBAUEAwUFBA...,there are two people that are standing on a tr...
1,1,Динамо,Q37996725,Екатеринбург,60.600349,56.845398,спорткомплекс в Екатеринбурге,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUEBAUEAwUFBA...,arafed building with a blue and white exterior...
2,2,Динамо,Q37996725,Екатеринбург,60.600349,56.845398,спорткомплекс в Екатеринбурге,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUEBAUEAwUFBA...,there is a blue and white building with a cloc...


## 3) EDA: что внутри датасета?




Нам важно понять:
- какие поля есть (название, город, описание, теги и т.п.);
- сколько пропусков/дубликатов;
- насколько длинные тексты (для выбора chunk_size).


In [60]:
data.info()


<class 'pandas.DataFrame'>
RangeIndex: 14634 entries, 0 to 14633
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   14634 non-null  int64  
 1   Name         14634 non-null  str    
 2   WikiData     12078 non-null  str    
 3   City         14634 non-null  str    
 4   Lon          14634 non-null  float64
 5   Lat          14634 non-null  float64
 6   description  12078 non-null  str    
 7   image        14634 non-null  str    
 8   en_txt       14634 non-null  str    
dtypes: float64(2), int64(1), str(6)
memory usage: 641.1 MB


In [61]:
# Быстрый взгляд на колонки и пропуски
na = data.isna().mean().sort_values(ascending=False)
display(pd.DataFrame({"na_rate": na}).head(20))



,na_rate
WikiData,0.174662
description,0.174662
Unnamed: 0,0.000000
City,0.000000
Name,0.000000
Lon,0.000000
Lat,0.000000
image,0.000000
en_txt,0.000000


In [62]:
# 10 примеров, где WikiData пустой
data[data["WikiData"].isna()].head(10)

,Unnamed: 0,Name,WikiData,City,Lon,Lat,description,image,en_txt
7761,1010,Мемориальная мастерская им. Французова,NaN,Владимир,40.398632,56.126774,NaN,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUEBAUEAwUFBA...,there is a desk with a chair and a picture on ...
7762,1011,Мемориальная мастерская им. Французова,NaN,Владимир,40.398632,56.126774,NaN,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUEBAUEAwUFBA...,there is a woman sitting in a room with a lot ...
7763,1012,Мемориальная мастерская им. Французова,NaN,Владимир,40.398632,56.126774,NaN,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUEBAUEAwUFBA...,there is a book shelf with a bunch of books on it
7764,1013,Мемориальная мастерская им. Французова,NaN,Владимир,40.398632,56.126774,NaN,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUEBAUEAwUFBA...,there is a drawing of a man sitting on a table
7765,1014,Мемориальная мастерская им. Французова,NaN,Владимир,40.398632,56.126774,NaN,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUEBAUEAwUFBA...,arafed view of a person walking down a street ...
7766,1015,Мемориальная мастерская им. Французова,NaN,Владимир,40.398632,56.126774,NaN,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUEBAUEAwUFBA...,a close up of a coin with a man's face on it
7767,1016,Мемориальная мастерская им. Французова,NaN,Владимир,40.398632,56.126774,NaN,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUEBAUEAwUFBA...,arafed plaque on a building with a man and a tree
7768,1017,Мемориальная мастерская им. Французова,NaN,Владимир,40.398632,56.126774,NaN,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUEBAUEAwUFBA...,arafed plaque on the side of a building with a...
7769,1018,Мемориальная мастерская им. Французова,NaN,Владимир,40.398632,56.126774,NaN,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUEBAUEAwUFBA...,there is a room with a lot of pictures on the ...
7770,1019,Мемориальная мастерская им. Французова,NaN,Владимир,40.398632,56.126774,NaN,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUEBAUEAwUFBA...,someone is holding a sheet of paper over a bic...


Мы хотим: **1 документ = 1 место**.

Для этого ввели понятие `place_id` — уникальный идентификатор места:

Если `WikiData` есть → это надёжный внешний ID (`Wikidata` entity), берём его.

Если `WikiData` нет → строим `fallback-ID` по данным о месте:
`City` + `Name` + (`Lat`, `Lon)` (с округлением координат).

Почему так:

- `City` + `Name` помогает различать одноимённые объекты в разных городах.

- Координаты фиксируют физическое положение объекта.

- Округление координат убирает микропогрешности (слегка разные значения у одного и того же места).

In [63]:
tmp = data.copy()

# 1) округляем координаты: убираем микропогрешности
tmp["Lat_r"] = tmp["Lat"].round(5)
tmp["Lon_r"] = tmp["Lon"].round(5)

# 2) place_id: если WikiData есть -> используем её, иначе fallback
tmp["place_id"] = np.where(
    tmp["WikiData"].notna(),
    "wd:" + tmp["WikiData"].astype(str),
    "fallback:" + tmp["City"].astype(str) + "|" + tmp["Name"].astype(str) + "|" +
    tmp["Lat_r"].astype(str) + "|" + tmp["Lon_r"].astype(str)
)

# 3) число уникальных мест
n_unique_places = tmp["place_id"].nunique()
n_unique_places

370

**Проверки, что логика разумная (sanity checks)**

Сделаем несколько оценок “уникальности”:

- `place_id` — наша целевая оценка уникальных POI

- `City+Name` — более “мягкая” (может считать разными то же место с разными координатами)

- `Lat/Lon` — географическая (может склеить разные объекты в одной точке)

In [64]:
n_by_place_id = tmp["place_id"].nunique()
n_by_city_name = tmp[["City","Name"]].drop_duplicates().shape[0]
n_by_coords = tmp[["Lat_r","Lon_r"]].drop_duplicates().shape[0]
n_by_place_id, n_by_city_name, n_by_coords

(370, 391, 399)

### Построение таблицы pois (POI-level dataset)

В исходном датасете одна и та же достопримечательность встречается много раз (разные фото/подписи/варианты описаний). Поэтому мы сначала ввели `place_id` - идентификатор места (POI, Point Of Interest), который группирует строки, относящиеся к одному объекту.

Таблица pois - это агрегированный датасет уровня “место”, где 1 строка = 1 уникальный POI.
Мы строим её так:

1. Берём `tmp` (сырые строки) и уже созданный `place_id`.

2. Группируем `tmp` по `place_id`.

3. Внутри каждой группы собираем “лучший” контент про место:

   - из `description` (RU) выбираем самое информативное описание (`description_best`);

   - из `en_txt` (EN подписи к фото) выбираем top-N наиболее полезных подписи и склеиваем в одну строку (`captions_top`);

   - формируем итоговый текст документа `doc_text`, который пойдёт в retrieval (BM25/dense) и дальше в RAG.

Шаг 1. Собираем POI-таблицу: схлопываем дубликаты по place_id, берём лучшее RU-описание и топ EN-капшены (для дальнейшей фильтрации/перевода). На этом шаге мы ещё не строим финальный retrieval-текст — только готовим поля.

In [65]:

TOP_CAPS = 8  # возьмём чуть больше, потом отфильтруем и урежем

POI_KWS = ["church","cathedral","monument","statue","tower","bridge","museum","park","kremlin","building"]

def caption_score(s: str) -> float:
    """Скор для капшенов: поощряем POI-слова, штрафуем шаблоны, но не рубим всё в ноль."""
    if not isinstance(s, str):
        return -1e9
    t = s.strip().lower()
    if len(t) < 20:
        return -1e9

    penalties = 0
    if t.startswith("there is") or t.startswith("there are"):
        penalties += 2
    if t.startswith("a close up") or "close up" in t:
        penalties += 2
    if "sitting" in t or "standing" in t:
        penalties += 1
    if "arafed" in t or "araff" in t:
        penalties += 1

    bonuses = 0
    for kw in POI_KWS:
        if kw in t:
            bonuses += 1

    return len(t) + 25*bonuses - 30*penalties

def agg_poi(g: pd.DataFrame) -> pd.Series:
    name = g["Name"].iloc[0]
    city = g["City"].iloc[0]
    lat = g["Lat"].iloc[0]
    lon = g["Lon"].iloc[0]
    wikidata = g["WikiData"].iloc[0]

    # RU best description
    descs = g["description"].dropna().astype(str).unique().tolist()
    desc_best = max(descs, key=len) if descs else ""
    has_desc = int(bool(desc_best))
    ru_len = len(desc_best)

    # EN captions candidates
    caps = g["en_txt"].dropna().astype(str).unique().tolist()
    n_caps_total = len(caps)
    caps_sorted = sorted(caps, key=caption_score, reverse=True)
    caps_top = caps_sorted[:TOP_CAPS]
    n_caps_used = len(caps_top)
    captions_top = " | ".join(caps_top)

    return pd.Series({
        "WikiData": wikidata,
        "Name": name,
        "City": city,
        "Lat": lat,
        "Lon": lon,
        "description_best": desc_best,
        "ru_len": ru_len,
        "captions_top": captions_top,
        "n_rows": len(g),
        "n_caps_total": n_caps_total,
        "n_caps_used": n_caps_used,
        "has_desc": has_desc,
    })

pois = tmp.groupby("place_id").apply(agg_poi).reset_index()
print(pois.shape)
pois.head(10)

(370, 13)


,place_id,WikiData,Name,City,Lat,Lon,description_best,ru_len,captions_top,n_rows,n_caps_total,n_caps_used,has_desc
0,fallback:Владимир|1941-1945|56.11119|40.36295,NaN,1941-1945,Владимир,56.111187,40.362949,,0,soldiers in a trench with a machine gun in fro...,30,30,8,0
1,fallback:Владимир|2000 летию Рождества Христов...,NaN,2000 летию Рождества Христова,Владимир,56.127674,40.408882,,0,cars are parked on the side of the road in fro...,58,58,8,0
2,fallback:Владимир|Аптекарь|56.12725|40.40231,NaN,Аптекарь,Владимир,56.127251,40.402309,,0,statue of a man holding a book and a microphon...,28,27,8,0
3,fallback:Владимир|Библиотечная лужайка|56.1264...,NaN,Библиотечная лужайка,Владимир,56.126423,40.402336,,0,children are reading books on a bench in a par...,55,55,8,0
4,fallback:Владимир|Бобёр|56.12108|40.37801,NaN,Бобёр,Владимир,56.121078,40.378010,,0,a beaver with an ax in his hand and a big axe ...,55,54,8,0
5,fallback:Владимир|Великая Княгиня Ольга|56.129...,NaN,Великая Княгиня Ольга,Владимир,56.129597,40.392128,,0,a picture of a painting of a woman and a man i...,59,56,8,0
6,fallback:Владимир|Венера|56.12268|40.38374,NaN,Венера,Владимир,56.122677,40.383736,,0,boy leaning against a tree in a park with a fr...,29,29,8,0
7,fallback:Владимир|Вечный огонь|56.12003|40.364,NaN,Вечный огонь,Владимир,56.120033,40.364002,,0,arafed monument in the center of a park with a...,31,30,8,0
8,fallback:Владимир|Виртуальный концертный зал|5...,NaN,Виртуальный концертный зал,Владимир,56.121613,40.384937,,0,orchestra playing a concert with a large scree...,29,29,8,0
9,fallback:Владимир|Владимир|56.10796|40.30917,NaN,Владимир,Владимир,56.107956,40.309170,,0,nighttime view of a city square with a large b...,38,34,8,0


In [66]:
has_ru = pois["description_best"].fillna("").astype(str).str.strip().ne("")
has_en = pois["captions_top"].fillna("").astype(str).str.strip().ne("")

pd.crosstab(has_ru, has_en, rownames=["has_ru_desc"], colnames=["has_en_caps"])

has_en_caps,True
has_ru_desc,
False,74
True,296


**Столбцы таблицы pois**

- `place_id` — уникальный ID места (POI). Используется как ключ группировки и связывания сущности:

   - wd:... если есть WikiData;

   - fallback:City|Name|Lat_r|Lon_r если WikiData отсутствует.

- `WikiData` — идентификатор Wikidata (`Q...`). `NaN`, если в исходных данных его нет.

- `Name` — название объекта/места.

- `City` — город.

- `Lat`, `Lon` — координаты места (широта/долгота), метаданные (могут быть полезны для запросов “что рядом”).

- `description_best` — лучшее русское описание места, выбранное из всех `description` внутри данного `place_id` (обычно берём самое длинное/информативное). Может быть пустым, если RU описаний нет.

- `captions_top` — строка с top-N английских подписей en_txt, склеенных через " | ". Это доп. контент по месту из подписей к фото.

- `n_rows` — сколько исходных строк было у этого места (сколько записей из raw схлопнулось в один POI).

- `n_caps_total` — сколько уникальных en_txt подписей было у места до отбора.

- `n_caps_used` — сколько подписей реально попало в `captions_top` (обычно = N, но может быть меньше).

- `has_desc` — флаг 0/1: есть ли у места русское описание `description_best`.

- `doc_len` — длина итогового текста `doc_text` (в символах). Полезно для диагностики/чистки (слишком короткие документы часто “пустые”).

- `doc_text` — финальный “документ места”, который индексируется в retrieval (BM25/dense) и используется в RAG как контекст.



Шаг 2. Диагностика данных: считаем, сколько POI имеют RU-описание, насколько оно длинное, и сколько POI нуждаются в “починке” (будем добивать русским переводом EN-капшенов).

In [67]:
print("POIs:", len(pois))
print("has_desc:", pois["has_desc"].value_counts(dropna=False).to_dict())
print("ru_len stats:\n", pois["ru_len"].describe())
NEED_RU_LEN = 65  # старт, потом потюним
need_ru = (pois["ru_len"] == 0) | (pois["ru_len"] < NEED_RU_LEN)
print("need_ru_fix (ru_len < 65):", int(need_ru.sum()))
pois.loc[need_ru, ["place_id","Name","City","ru_len","captions_top"]].head(15)

POIs: 370
has_desc: {1: 296, 0: 74}
ru_len stats:
 count    370.000000
mean      26.572973
std       23.313611
min        0.000000
25%       16.000000
50%       26.000000
75%       35.000000
max      225.000000
Name: ru_len, dtype: float64
need_ru_fix (ru_len < 65): 357


,place_id,Name,City,ru_len,captions_top
0,fallback:Владимир|1941-1945|56.11119|40.36295,1941-1945,Владимир,0,soldiers in a trench with a machine gun in fro...
1,fallback:Владимир|2000 летию Рождества Христов...,2000 летию Рождества Христова,Владимир,0,cars are parked on the side of the road in fro...
2,fallback:Владимир|Аптекарь|56.12725|40.40231,Аптекарь,Владимир,0,statue of a man holding a book and a microphon...
3,fallback:Владимир|Библиотечная лужайка|56.1264...,Библиотечная лужайка,Владимир,0,children are reading books on a bench in a par...
4,fallback:Владимир|Бобёр|56.12108|40.37801,Бобёр,Владимир,0,a beaver with an ax in his hand and a big axe ...
5,fallback:Владимир|Великая Княгиня Ольга|56.129...,Великая Княгиня Ольга,Владимир,0,a picture of a painting of a woman and a man i...
6,fallback:Владимир|Венера|56.12268|40.38374,Венера,Владимир,0,boy leaning against a tree in a park with a fr...
7,fallback:Владимир|Вечный огонь|56.12003|40.364,Вечный огонь,Владимир,0,arafed monument in the center of a park with a...
8,fallback:Владимир|Виртуальный концертный зал|5...,Виртуальный концертный зал,Владимир,0,orchestra playing a concert with a large scree...
9,fallback:Владимир|Владимир|56.10796|40.30917,Владимир,Владимир,0,nighttime view of a city square with a large b...


## 4) Чистка и удаление "мусорных" строк

**Задача:** убрать явные выбросы/мусор, иначе векторный поиск и RAG будут тащить в контекст не то.

После ввода `place_id` и агрегации в таблицу `pois` (1 строка = 1 POI) чистку будем делать на уровне POI-документов.


Ниже — два слоя очистки:

1. Sanity-фильтры по длинам (быстро отсекают пустые/битые документы).

2. TF-IDF outlier detection + audit (выбираем подозрительные POI, быстро просматриваем и удаляем только то, что реально мусор)


In [68]:
# Чистим EN-captions на уровне POI

# Идея:
# - У каждого POI есть набор EN подписей к фото.
# - Там много шаблонного мусора (people / phone / cars parked / close up / arafed и т.п.).
# - Мы НЕ удаляем POI, а только выбираем из его подписей небольшой набор "лучших" строк:
#   * captions_good: то, что похоже на описание достопримечательности/места
#   * captions_fb: fallback (если хороших 0) — берём несколько самых длинных НЕ-ужасных
#
# Далее мы будем переводить на русский только выбранные строки (чтобы не переводить мусор).


TOP_CAPS_GOOD = 5          # сколько "хороших" оставляем
TOP_CAPS_FALLBACK = 2      # если хороших 0 — сколько берём в fallback
MIN_GOOD_SCORE = 60        # порог "хорошести" (пока фиксируем, позже можно тюнить)

# Нормализация: режем "arafed/araffatured", лишние пробелы, приводим к lower
_ARAFED_RE = re.compile(r"\b(arafed|araffatured|araffature|arafed)\b", re.I)

def normalize_en(s: str) -> str:
    if not isinstance(s, str):
        return ""
    t = s.strip()
    t = _ARAFED_RE.sub("", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

# Слова/фразы, которые часто означают "не про место" (люди/телефоны/машины/реклама)
BAD_SUBSTR = [
    "cars parked", "car parked", "cell phone", "selfie", "logo",
    "a person holding", "people standing", "people sitting",
    "in front of a building", "in a room", "on a street", "car","cars","parked","parking","traffic","road","street","sidewalk",
    "phone","cell phone","smartphone","selfie","laptop",
    "man","woman","boy","girl","people","person","crowd",
    "dog","cat","bear","animal",
    "logo","banner","poster","text","advertisement","ad",
    "close up","close-up","нннн"
]

# Слова, которые чаще встречаются в "место-ориентированных" описаниях
GOOD_KW = [
    "church","cathedral","chapel","monastery","temple",
    "museum","gallery","exhibition",
    "monument","memorial","statue","sculpture",
    "tower","belltower","clock tower",
    "bridge","river","embankment",
    "park","garden","square","street","avenue",
    "kremlin","fortress","wall","gate",
    "building","house","mansion","palace", "church","cathedral","chapel","monastery","kremlin","temple",
    "museum","gallery","theatre","tower","bell tower","dome",
    "bridge","embankment","monument","memorial","statue","sculpture",
    "park","garden","square","palace","mansion","castle","fortress",
    "building","facade","architecture","historic","heritage"
]

def caption_score(s: str) -> float:
    '''
    Скоринг подписей:
    + бонусы за GOOD_KW (похоже на объект/архитектуру)
    - штрафы за шаблоны/мусор (people/phone/cars/close up)
    Итог: больше = лучше. Отрицательные — явный мусор.
    '''
    if not isinstance(s, str):
        return -1e9
    t = normalize_en(s).lower()
    if len(t) < 15:
        return -1e9

    score = len(t)  # базово: длиннее — чуть информативнее

    # мягкие штрафы за шаблоны
    if t.startswith("there is") or t.startswith("there are"):
        score -= 40
    if "close up" in t or t.startswith("a close up"):
        score -= 35
    if "sitting" in t or "standing" in t:
        score -= 20

    # штраф за мусорные подстроки
    for bad in BAD_SUBSTR:
        if bad in t:
            score -= 25

    # бонусы за "место-слова"
    for kw in GOOD_KW:
        if kw in t:
            score += 20

    return score

def filter_captions(caps, top_n=TOP_CAPS_GOOD, min_score=MIN_GOOD_SCORE):
    '''
    Возвращает (good, fallback):
    - good: top_n капшенов со score >= min_score
    - fallback: если good пустой, берём TOP_CAPS_FALLBACK лучших со score >= 0 (просто не явный мусор)
    '''
    caps = [normalize_en(c) for c in (caps or [])]
    caps = [c for c in caps if c]  # remove empty

    # unique stable (case-insensitive)
    seen=set(); uniq=[]
    for c in caps:
        lc=c.lower()
        if lc not in seen:
            seen.add(lc); uniq.append(c)

    scored = [(c, caption_score(c)) for c in uniq]
    scored.sort(key=lambda x: x[1], reverse=True)

    good = [c for c,sc in scored if sc >= min_score][:top_n]

    fallback = []
    if not good:
        fallback = [c for c,sc in scored if sc >= 0][:TOP_CAPS_FALLBACK]

    return good, fallback

def split_caps_top(s: str):
    # captions_top у тебя строкой "cap1 | cap2 | ..."
    if not isinstance(s, str) or not s.strip():
        return []
    return [x.strip() for x in s.split("|") if x.strip()]

# Собираем в pois2 очищенные captions
pois2 = pois.copy()
pois2["ru_text"] = pois2["description_best"].fillna("").astype(str).str.strip()
pois2["ru_len"] = pois2["ru_text"].str.len()

def _apply_caps_row(row):
    caps = split_caps_top(row.get("captions_top", ""))
    good, fb = filter_captions(caps)
    return pd.Series({
        "captions_good_list": good,
        "captions_fb_list": fb,
        "n_caps_good": len(good),
        "n_caps_fb": len(fb),
        "captions_good": " | ".join(good),
        "captions_fb": " | ".join(fb),
    })

pois2 = pd.concat([pois2, pois2.apply(_apply_caps_row, axis=1)], axis=1)

print("POIs:", len(pois2))
print("POI with >=1 good caps:", int((pois2["n_caps_good"] >= 1).sum()))
print("POI with 0 good caps:", int((pois2["n_caps_good"] == 0).sum()))


POIs: 370
POI with >=1 good caps: 360
POI with 0 good caps: 10


### Чистим EN-captions для всех POI

Мы **не удаляем POI**, а приводим EN подписи к виду “можно использовать как подсказку”:

- `captions_good` — top-5 строк, которые реально похожи на описание места/объекта (по скорингу).
- `captions_fb` — fallback: если `captions_good` пустой, берём несколько “не ужасных” строк.

Дальше перевод будем делать только по выбранным строкам, чтобы не переводить мусор.


In [69]:
# Быстрые проверки качества скоринга
# 1) Посмотрим худшие/лучшие подписи глобально (по уникальным caption из captions_top)

all_caps = []
for s in pois2["captions_top"].fillna("").astype(str):
    all_caps.extend(split_caps_top(s))

all_caps = [normalize_en(c) for c in all_caps]
all_caps = [c for c in all_caps if c]

# unique (case-insensitive), сохраним первое вхождение
orig_map={}
for c in all_caps:
    lc=c.lower()
    if lc not in orig_map:
        orig_map[lc]=c

caps_scored = [(c, caption_score(c)) for c in orig_map.values()]
caps_scored.sort(key=lambda x: x[1])

print("Total unique caps:", len(caps_scored))

print("\nWorst 20 (скор маленький/отрицательный => мусор):")
for c, sc in caps_scored[:20]:
    print(f"{sc:>7.1f} | {c}")

print("\nBest 20 (скор большой => похоже на место/объект):")
for c, sc in caps_scored[-20:][::-1]:
    print(f"{sc:>7.1f} | {c}")

# 2) Посмотрим POI, где хороших подписей мало/много
print("\nPOI with n_caps_good == 0:", int((pois2['n_caps_good']==0).sum()))
print("POI with n_caps_good >= 1:", int((pois2['n_caps_good']>=1).sum()))

display_cols = ["place_id","Name","City","ru_len","n_caps_good","captions_good","captions_fb"]
print("\nWorst (no good caps) sample:")
display(pois2.loc[pois2["n_caps_good"]==0, display_cols].head(10))

print("\nBest (many good caps) sample:")
display(pois2.sort_values("n_caps_good", ascending=False)[display_cols].head(10))


Total unique caps: 2513

Worst 20 (скор маленький/отрицательный => мусор):
  -58.0 | crowd of people taking pictures of a concert with their cell phones
  -46.0 | araful looking street scene with cars and buses on the road
  -42.0 | there is a car parked on the side of the road in front of a building
  -39.0 | cars parked on the side of a road in front of a building
  -37.0 | cars parked on the side of the road in front of a building
  -33.0 | cars parked on the side of a snowy road in front of a building
  -26.0 | image of a stadium with a lot of cars parked in front of it
  -26.0 | people looking at stuffed animals in a display case with a woman and child
  -25.0 | aerial view of a parking lot with a lot of cars parked in it
  -24.0 | snowy street with a fence and a car parked on the side of the road
  -23.0 | a painting of a woman reading a book in a white veil
  -23.0 | cars parked on the side of a road near a church
  -22.0 | cars parked in front of a blue house on a street
  -22.

,place_id,Name,City,ru_len,n_caps_good,captions_good,captions_fb
5,fallback:Владимир|Великая Княгиня Ольга|56.129...,Великая Княгиня Ольга,Владимир,0,0,,a picture of a painting of a woman and a man i...
28,fallback:Владимир|Кино 9D|56.12903|40.40342,Кино 9D,Владимир,0,0,,people wearing virtual glasses are watching a ...
37,fallback:Владимир|Мемориальная мастерская им. ...,Мемориальная мастерская им. Французова,Владимир,0,0,,plaque on a building with a man and a tree | t...
45,fallback:Владимир|Огни Владимира|56.12762|40.4...,Огни Владимира,Владимир,0,0,,woman in a museum looking at a display of glas...
46,fallback:Владимир|П.И. Чайковский|56.12208|40....,П.И. Чайковский,Владимир,0,0,,painting of a man standing in front of a tree ...
59,fallback:Владимир|Танкистам|56.14725|40.31762,Танкистам,Владимир,0,0,,soldiers pose in front of a tank in a city | t...
146,wd:Q2503505,Локомотив,Нижний Новгород,47,0,,soccer players in action on a soccer field dur...
274,wd:Q4476708,Уральский государственный медицинский университет,Екатеринбург,40,0,,cars are parked in front of a large beige buil...
356,wd:Q93622482,Шахматный дом (Игорный дом А.И. Троицкого),Нижний Новгород,41,0,,cars parked in front of a pink building with a...
366,wd:Q98381215,Дом архитектора,Нижний Новгород,49,0,,cars parked in front of a pink building with a...



Best (many good caps) sample:


,place_id,Name,City,ru_len,n_caps_good,captions_good,captions_fb
369,wd:Q99929827,Дом Сироткина,Нижний Новгород,68,5,view of a large white building with a clock to...,
368,wd:Q99622643,Дворец вице-губернатора,Нижний Новгород,50,5,an old photo of a building with a clock tower ...,
1,fallback:Владимир|2000 летию Рождества Христов...,2000 летию Рождества Христова,Владимир,0,5,monument in a snowy park with a statue in the ...,
2,fallback:Владимир|Аптекарь|56.12725|40.40231,Аптекарь,Владимир,0,5,yellow and green building with a statue in fro...,
367,wd:Q99522676,Рекорд,Нижний Новгород,71,5,view of a building with a lot of windows and a...,
365,wd:Q97999957,Здание Удельной конторы,Нижний Новгород,23,5,black and white photograph of a building with ...,
364,wd:Q97678623,Церковь в честь Рождества Богородицы,Нижний Новгород,43,5,view of a church with a clock tower and a rive...,
363,wd:Q97660371,Дом В. И. Смирнова,Нижний Новгород,18,5,wooden house with a clock tower and a steeple ...,
7,fallback:Владимир|Вечный огонь|56.12003|40.364,Вечный огонь,Владимир,0,5,view of a large white building with a clock to...,
362,wd:Q97591769,Комплекс штаба военного округа,Екатеринбург,30,5,view of a large building with a statue in fron...,


### Перевод выбранных EN-captions → RU (offline)

Перевод делаем **один раз при подготовке базы**. В проде перевод на лету почти всегда плохая идея:
- дорого и медленно,
- усложняет систему,
- добавляет точки отказа.

Если модель перевода не доступна (не скачалась/ошибка), пайплайн не ломаем: просто пропускаем перевод.


In [70]:
# Перевод выбранных EN-captions -> RU (офлайн предобработка)

import torch
from transformers import MarianMTModel, MarianTokenizer

MODEL_NAME = "Helsinki-NLP/opus-mt-en-ru"

def load_translator():
    try:
        tok = MarianTokenizer.from_pretrained(MODEL_NAME)
        mdl = MarianMTModel.from_pretrained(MODEL_NAME)
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        mdl = mdl.to(dev)
        return tok, mdl, dev, None
    except Exception as e:
        return None, None, None, e

tokenizer, model, device, trans_err = load_translator()
if trans_err:
    print("Translator NOT available, will fallback (no translation). Reason:", repr(trans_err))
else:
    print("Translator OK. Device:", device)

def translate_en_ru(texts, batch_size=16, max_length=256):
    if not texts:
        return []
    if model is None:
        return [""] * len(texts)

    outs=[]
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        batch = [t if isinstance(t,str) else "" for t in batch]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)
        with torch.no_grad():
            gen = model.generate(**inputs, max_length=max_length)
        out = tokenizer.batch_decode(gen, skip_special_tokens=True)
        outs.extend(out)
    return outs

def caps_en_used(row):
    caps = row.get("captions_good_list", []) or []
    if len(caps) == 0:
        caps = row.get("captions_fb_list", []) or []
    return caps

pois2["caps_en_used_list"] = pois2.apply(caps_en_used, axis=1)

# Уникальный список для перевода + кеш
uniq_to_translate = []
seen=set()
for caps in pois2["caps_en_used_list"]:
    for c in caps:
        lc=c.lower()
        if lc not in seen:
            seen.add(lc)
            uniq_to_translate.append(c)

print("Unique EN strings to translate:", len(uniq_to_translate))

ru_trans = translate_en_ru(uniq_to_translate, batch_size=16)
trans_map = {en.lower(): ru for en,ru in zip(uniq_to_translate, ru_trans)}

def map_caps_to_ru(caps):
    outs=[]
    for c in caps:
        ru = trans_map.get(c.lower(), "")
        if isinstance(ru, str) and ru.strip():
            outs.append(ru.strip())
    return outs

pois2["caps_ru_used_list"] = pois2["caps_en_used_list"].apply(map_caps_to_ru)
pois2["caps_ru_used"] = pois2["caps_ru_used_list"].apply(lambda xs: " | ".join(xs))

print("POI with >=1 translated cap:", int((pois2["caps_ru_used"].str.len() > 0).sum()))


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Translator OK. Device: cuda
Unique EN strings to translate: 1421
POI with >=1 translated cap: 370


### Финальная сборка `retr_text_ru` (строго русский текст для поиска)

Правило (RU_MIN_LEN = 60):
- `ru_len >= 60` → используем RU описание (и шапку `Name + City`).
- `0 < ru_len < 60` → RU + добивка переведёнными капшенами (если они есть).
- `ru_len == 0` → используем только переведённые капшены (если есть), иначе только `Name + City`.


In [71]:
# Финальная сборка retr_text_ru (строго RU для поиска)


RU_MIN_LEN = 60

def build_retr_text_ru(row):
    name = (row.get("Name") or "").strip()
    city = (row.get("City") or "").strip()
    header = f"{name}. Город: {city}."
    ru = (row.get("ru_text") or "").strip()
    ru_len = int(row.get("ru_len") or 0)
    caps_ru = (row.get("caps_ru_used") or "").strip()

    if ru_len >= RU_MIN_LEN:
        return header + (" " + ru if ru else ""), "RU_OK"
    if 0 < ru_len < RU_MIN_LEN:
        if caps_ru:
            return header + " " + ru + " Описание по фото: " + caps_ru, "RU_SHORT_PLUS_CAPS"
        else:
            return header + (" " + ru if ru else ""), "RU_SHORT_NO_CAPS"
    if caps_ru:
        return header + " Описание по фото: " + caps_ru, "NO_RU_USE_CAPS"
    return header, "NO_RU_NO_CAPS"

tmp = pois2.apply(lambda r: build_retr_text_ru(r), axis=1, result_type="expand")
pois2["retr_text_ru"] = tmp[0]
pois2["rule_used"] = tmp[1]
pois2["retr_len"] = pois2["retr_text_ru"].str.len()

print("Rules distribution:")
print(pois2["rule_used"].value_counts())

print("\nretr_len stats:")
print(pois2["retr_len"].describe())

# Примеры
for rule in ["RU_OK","RU_SHORT_PLUS_CAPS","NO_RU_USE_CAPS","NO_RU_NO_CAPS"]:
    sub = pois2[pois2["rule_used"]==rule].head(3)
    if len(sub)==0:
        continue
    print("\n" + "="*90)
    print("Rule:", rule, " | total:", int((pois2["rule_used"]==rule).sum()))
    for _,row in sub.iterrows():
        print(f"\n[{rule}] {row['Name']} — {row['City']} | place_id={row['place_id']}")
        print(row["retr_text_ru"][:400], "..." if len(row["retr_text_ru"])>400 else "")


Rules distribution:
rule_used
RU_SHORT_PLUS_CAPS    281
NO_RU_USE_CAPS         74
RU_OK                  15
Name: count, dtype: int64

retr_len stats:
count    370.000000
mean     319.418919
std       82.513333
min       91.000000
25%      293.500000
50%      341.500000
75%      373.000000
max      492.000000
Name: retr_len, dtype: float64

Rule: RU_OK  | total: 15

[RU_OK] Дом-музей Столетовых — Владимир | place_id=wd:Q107422934
Дом-музей Столетовых. Город: Владимир. Мемориальный музей в городе Владимире, посвящённый жизни и деятельности выдающегося учёного-физика А. Г. Столетова и генерала от инфантерии Н. Г. Столетова, сыгравшего видную роль в освобождении Болгарии от турецкого ига 

[RU_OK] Театр оперы и балета — Нижний Новгород | place_id=wd:Q172218
Театр оперы и балета. Город: Нижний Новгород. Нижегородский государственный академический театр оперы и балета имени А. С. Пушкина 

[RU_OK] Дмитриевская башня — Нижний Новгород | place_id=wd:Q18406802
Дмитриевская башня. Город: Нижний

### TF-IDF sanity check (по `retr_text_ru`)

На этом шаге мы делаем **быструю проверку качества собранных русских текстов для ретривала** (`retr_text_ru`) после всех правил:
- где RU-описание нормальное (>= RU_MIN_LEN) — оставили его,
- где RU короткое — “добили” переводом отфильтрованных EN-captions,
- где RU нет — используем перевод caps (или fallback) + Name/City.

**Идея TF-IDF тут**:
мы строим TF-IDF вектора для всех `retr_text_ru`, считаем **cosine similarity** между всеми POI и для каждого POI считаем `tfidf_mean_sim` — среднюю похожесть на остальные.

- **Низкая `tfidf_mean_sim`** → объект “не похож” на остальные, часто это признак:
  - мусорных/нерелевантных captions,
  - странного перевода,
  - слишком общего текста (“вид города”, “машины припаркованы…”),
  - технических артефактов.

Дальше мы **смотрим топ-кандидатов** с самой низкой похожестью и решаем:
- подкрутить фильтр captions / порог,
- добавить стоп-слова,
- вручную поправить несколько проблемных POI,
- или оставить как есть, если выглядит ок.

TF-IDF — это **контроль качества** и поиск выбросов, а не обязательное удаление данных.

In [72]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Предполагаем, что у тебя уже есть pois2 и колонка retr_text_ru
df = pois2.copy()

assert "retr_text_ru" in df.columns, "Нет колонки retr_text_ru в pois2"
texts = df["retr_text_ru"].fillna("").astype(str)

# Чтобы TF-IDF не упал, если вдруг есть пустые строки
non_empty_mask = texts.str.strip().str.len() > 0
df_ne = df[non_empty_mask].copy()
texts_ne = texts[non_empty_mask].copy()

print("All POIs:", len(df), "| non-empty retr_text_ru:", len(df_ne), "| empty:", int((~non_empty_mask).sum()))

# TF-IDF
vectorizer = TfidfVectorizer(
    min_df=2,           # термин должен встретиться хотя бы в 2 документах
    max_df=0.90,        # слишком частые термины отсекаем
    ngram_range=(1, 2), # униграммы + биграммы
    lowercase=True
)

X = vectorizer.fit_transform(texts_ne)

# cosine similarity матрица (370x370 ок)
S = cosine_similarity(X)

# средняя похожесть каждого документа на остальные (без себя)
mean_sim = (S.sum(axis=1) - 1) / (S.shape[0] - 1)
df_ne["tfidf_mean_sim"] = mean_sim

# вернем значения обратно в общий df (для пустых поставим NaN)
df["tfidf_mean_sim"] = np.nan
df.loc[df_ne.index, "tfidf_mean_sim"] = df_ne["tfidf_mean_sim"]

# кандидаты-выбросы (самые "непохожие")
TOP_N = 40
candidates = df_ne.sort_values("tfidf_mean_sim").head(TOP_N)

cols_to_show = [c for c in ["place_id", "Name", "City", "ru_len", "retr_len", "tfidf_mean_sim"] if c in df.columns]
if "tfidf_mean_sim" not in cols_to_show:
    cols_to_show.append("tfidf_mean_sim")

display(candidates[cols_to_show])

# ---- (опционально) быстро посмотреть "почему так": топ TF-IDF термины для нескольких кандидатов
feature_names = np.array(vectorizer.get_feature_names_out())

def top_terms_for_row(row_idx, top_k=12):
    v = X[row_idx]
    if v.nnz == 0:
        return []
    inds = v.indices
    vals = v.data
    order = np.argsort(vals)[::-1][:top_k]
    return list(zip(feature_names[inds[order]], vals[order]))

print("\nTop terms for first 5 outliers:")
for i, (idx, row) in enumerate(candidates.head(5).iterrows()):
    # row position in df_ne/X:
    row_pos = np.where(df_ne.index.values == idx)[0][0]
    print(f"\n[{i}] {row.get('Name','')} — {row.get('City','')} | mean_sim={row['tfidf_mean_sim']:.4f}")
    print("Top terms:", top_terms_for_row(row_pos, top_k=12))
    print("retr_text_ru preview:", str(row["retr_text_ru"])[:220], "...")

All POIs: 370 | non-empty retr_text_ru: 370 | empty: 0


,place_id,Name,City,ru_len,retr_len,tfidf_mean_sim
177,wd:Q4091101,Больница имени Н. В. Соловьёва,Ярославль,76,126,0.009051
235,wd:Q4318618,Нижегородская государственная консерватория им...,Нижний Новгород,62,147,0.011439
115,wd:Q172218,Театр оперы и балета,Нижний Новгород,84,130,0.011942
54,fallback:Владимир|Сердце|56.12999|40.41028,Сердце,Владимир,0,91,0.013293
239,wd:Q4318683,Нижегородское речное училище,Нижний Новгород,225,279,0.013406
136,wd:Q2036269,Шарташские каменные палатки,Екатеринбург,64,114,0.015342
208,wd:Q4174330,Свердловское художественное училище им. И. Д. ...,Екатеринбург,55,195,0.015655
45,fallback:Владимир|Огни Владимира|56.12762|40.4...,Огни Владимира,Владимир,0,164,0.015696
264,wd:Q4410161,Свердловский областной краеведческий музей,Екатеринбург,42,203,0.016512
339,wd:Q775487,Собор Успения Пресвятой Богородицы,Владимир,83,136,0.018474



Top terms for first 5 outliers:

[0] Больница имени Н. В. Соловьёва — Ярославль | mean_sim=0.0091
Top terms: [('больница', np.float64(0.7751860474469731)), ('имени', np.float64(0.5904641801182502)), ('город ярославль', np.float64(0.15880592533155338)), ('ярославль', np.float64(0.15880592533155338))]
retr_text_ru preview: Больница имени Н. В. Соловьёва. Город: Ярославль. ГАУЗ ЯО Клиническая больница скорой медицинской помощи имени Н. В. Соловьёва ...

[1] Нижегородская государственная консерватория им. М.И. Глинки — Нижний Новгород | mean_sim=0.0114
Top terms: [('государственная консерватория', np.float64(0.47447112925601626)), ('консерватория', np.float64(0.47447112925601626)), ('государственная', np.float64(0.4510083477456451)), ('нижегородская', np.float64(0.4328092005227637)), ('новгород нижегородская', np.float64(0.22550417387282254)), ('им', np.float64(0.19243524065587556)), ('имени', np.float64(0.18070384990069)), ('город нижний', np.float64(0.09808749354829942)), ('нижний новго

### Отбор кандидатов для аудита

После расчёта TF-IDF similarity выбираем объекты с наименьшей средней похожестью. Именно они являются первыми кандидатами на ручную проверку.

На этом этапе мы не удаляем объекты автоматически, а только формируем shortlist для последующего анализа.

In [73]:
TOP_N = 370
candidates = df_ne.sort_values("tfidf_mean_sim").head(TOP_N)

cols = [c for c in ["place_id","Name","City","ru_len","retr_len","tfidf_mean_sim","rule_used"] if c in candidates.columns]
display(candidates[cols])

,place_id,Name,City,ru_len,retr_len,tfidf_mean_sim,rule_used
177,wd:Q4091101,Больница имени Н. В. Соловьёва,Ярославль,76,126,0.009051,RU_OK
235,wd:Q4318618,Нижегородская государственная консерватория им...,Нижний Новгород,62,147,0.011439,RU_OK
115,wd:Q172218,Театр оперы и балета,Нижний Новгород,84,130,0.011942,RU_OK
54,fallback:Владимир|Сердце|56.12999|40.41028,Сердце,Владимир,0,91,0.013293,NO_RU_USE_CAPS
239,wd:Q4318683,Нижегородское речное училище,Нижний Новгород,225,279,0.013406,RU_OK
...,...,...,...,...,...,...,...
130,wd:Q1984395,№16 Усадьба Расторгуева-Харитонова,Екатеринбург,22,408,0.131611,RU_SHORT_PLUS_CAPS
9,fallback:Владимир|Владимир|56.10796|40.30917,Владимир,Владимир,0,326,0.132229,NO_RU_USE_CAPS
11,fallback:Владимир|Владимир|56.15805|40.36194,Владимир,Владимир,0,326,0.132229,NO_RU_USE_CAPS
10,fallback:Владимир|Владимир|56.15245|40.36764,Владимир,Владимир,0,326,0.132229,NO_RU_USE_CAPS


### Ручной просмотр подозрительных объектов

Здесь выводим подозрительные записи в удобном текстовом виде: название, город, `place_id`, среднюю similarity и фрагмент `retr_text_ru`.

Цель шага — быстро глазами понять, действительно ли объект шумный, нерелевантный или просто редкий, но валидный.

In [74]:
def show_candidate(df, k=40):
    for i, row in df.head(k).iterrows():
        print("="*110)
        print(f"{row.get('Name')} — {row.get('City')} | mean_sim={row['tfidf_mean_sim']:.4f} | place_id={row.get('place_id')}")
        if "rule_used" in row:
            print("rule_used:", row["rule_used"])
        print("retr_text_ru:\n", row["retr_text_ru"][:800])

show_candidate(candidates, k=40)

Больница имени Н. В. Соловьёва — Ярославль | mean_sim=0.0091 | place_id=wd:Q4091101
rule_used: RU_OK
retr_text_ru:
 Больница имени Н. В. Соловьёва. Город: Ярославль. ГАУЗ ЯО Клиническая больница скорой медицинской помощи имени Н. В. Соловьёва
Нижегородская государственная консерватория им. М.И. Глинки — Нижний Новгород | mean_sim=0.0114 | place_id=wd:Q4318618
rule_used: RU_OK
retr_text_ru:
 Нижегородская государственная консерватория им. М.И. Глинки. Город: Нижний Новгород. Нижегородская государственная консерватория имени М. И. Глинки
Театр оперы и балета — Нижний Новгород | mean_sim=0.0119 | place_id=wd:Q172218
rule_used: RU_OK
retr_text_ru:
 Театр оперы и балета. Город: Нижний Новгород. Нижегородский государственный академический театр оперы и балета имени А. С. Пушкина
Сердце — Владимир | mean_sim=0.0133 | place_id=fallback:Владимир|Сердце|56.12999|40.41028
rule_used: NO_RU_USE_CAPS
retr_text_ru:
 Сердце. Город: Владимир. Описание по фото: Кто-то держал пару очков в руках с кучей н

### Грубая эвристика для пометки объектов на удаление

После ручного просмотра вводим простую rule-based эвристику `suggest_action(...)`, которая помогает разделить найденные записи на:
- потенциально нерелевантные (`DROP`);
- сомнительные, но требующие дополнительной проверки;
- валидные записи.



In [75]:
audit = candidates.copy()

def suggest_action(name, text):
    t = (str(name) + " " + str(text)).lower()
    # очень грубые правила — только чтобы быстро разделить
    if any(x in t for x in ["больница", "Лицей", "поликлиника", "аптека", "колледж", "сердце", "лужайка", "венера", "школа", "лицей", "на фасаде", "детский сад", "техникум", "филиал", "кладбище", "Венера", "кино" , "Панно"]):
        return "DROP"
    if any(x in t for x in ["кто-то", "держал", "селфи", "пара очков", "машины припаркованы", "вид города"]):
        return "FIX_CAPS"

    if any(x in t for x in ["музей", "памятник", "усадьба", "монастырь", "театр", "башня", "храм", "собор", "церковь"]):
      return "OK"
    return "OK"

audit["action_suggest"] = audit.apply(lambda r: suggest_action(r.get("Name",""), r.get("retr_text_ru","")), axis=1)
audit[["place_id","Name","City","tfidf_mean_sim","action_suggest"]].head(40)

,place_id,Name,City,tfidf_mean_sim,action_suggest
177,wd:Q4091101,Больница имени Н. В. Соловьёва,Ярославль,0.009051,DROP
235,wd:Q4318618,Нижегородская государственная консерватория им...,Нижний Новгород,0.011439,OK
115,wd:Q172218,Театр оперы и балета,Нижний Новгород,0.011942,OK
54,fallback:Владимир|Сердце|56.12999|40.41028,Сердце,Владимир,0.013293,DROP
239,wd:Q4318683,Нижегородское речное училище,Нижний Новгород,0.013406,DROP
136,wd:Q2036269,Шарташские каменные палатки,Екатеринбург,0.015342,OK
208,wd:Q4174330,Свердловское художественное училище им. И. Д. ...,Екатеринбург,0.015655,OK
45,fallback:Владимир|Огни Владимира|56.12762|40.4...,Огни Владимира,Владимир,0.015696,OK
264,wd:Q4410161,Свердловский областной краеведческий музей,Екатеринбург,0.016512,OK
339,wd:Q775487,Собор Успения Пресвятой Богородицы,Владимир,0.018474,OK


In [76]:
drop_ids = audit.loc[audit["action_suggest"] == "DROP", "place_id"].unique().tolist()
len(drop_ids), drop_ids[:10]

(25,
 ['wd:Q4091101',
  'fallback:Владимир|Сердце|56.12999|40.41028',
  'wd:Q4318683',
  'wd:Q99522676',
  'fallback:Владимир|Панно на фасаде|56.13826|40.36419',
  'fallback:Владимир|Кино 9D|56.12903|40.40342',
  'fallback:Владимир|Библиотечная лужайка|56.12642|40.40234',
  'fallback:Владимир|Венера|56.12268|40.38374',
  'wd:Q4538877',
  'wd:Q4122453'])

In [77]:
drop_view = audit.loc[audit["action_suggest"] == "DROP",
                      ["place_id","Name","City","tfidf_mean_sim","action_suggest","retr_text_ru"]].copy()

drop_view["retr_preview"] = drop_view["retr_text_ru"].astype(str).str.slice(0, 220)

drop_view.sort_values("tfidf_mean_sim").head(30)

,place_id,Name,City,tfidf_mean_sim,action_suggest,retr_text_ru,retr_preview
177,wd:Q4091101,Больница имени Н. В. Соловьёва,Ярославль,0.009051,DROP,Больница имени Н. В. Соловьёва. Город: Ярослав...,Больница имени Н. В. Соловьёва. Город: Ярослав...
54,fallback:Владимир|Сердце|56.12999|40.41028,Сердце,Владимир,0.013293,DROP,Сердце. Город: Владимир. Описание по фото: Кто...,Сердце. Город: Владимир. Описание по фото: Кто...
239,wd:Q4318683,Нижегородское речное училище,Нижний Новгород,0.013406,DROP,Нижегородское речное училище. Город: Нижний Но...,Нижегородское речное училище. Город: Нижний Но...
367,wd:Q99522676,Рекорд,Нижний Новгород,0.018627,DROP,Рекорд. Город: Нижний Новгород. Общежитие инст...,Рекорд. Город: Нижний Новгород. Общежитие инст...
49,fallback:Владимир|Панно на фасаде|56.13826|40....,Панно на фасаде,Владимир,0.026591,DROP,Панно на фасаде. Город: Владимир. Описание по ...,Панно на фасаде. Город: Владимир. Описание по ...
28,fallback:Владимир|Кино 9D|56.12903|40.40342,Кино 9D,Владимир,0.028861,DROP,Кино 9D. Город: Владимир. Описание по фото: Лю...,Кино 9D. Город: Владимир. Описание по фото: Лю...
3,fallback:Владимир|Библиотечная лужайка|56.1264...,Библиотечная лужайка,Владимир,0.032517,DROP,Библиотечная лужайка. Город: Владимир. Описани...,Библиотечная лужайка. Город: Владимир. Описани...
6,fallback:Владимир|Венера|56.12268|40.38374,Венера,Владимир,0.033109,DROP,Венера. Город: Владимир. Описание по фото: зна...,Венера. Город: Владимир. Описание по фото: зна...
301,wd:Q4538877,Ярославский дворец молодёжи,Ярославль,0.035102,DROP,Ярославский дворец молодёжи. Город: Ярославль....,Ярославский дворец молодёжи. Город: Ярославль....
187,wd:Q4122453,Воинское мемориальное кладбище,Ярославль,0.041587,DROP,Воинское мемориальное кладбище. Город: Ярослав...,Воинское мемориальное кладбище. Город: Ярослав...


In [78]:
pois2_clean = pois2[~pois2["place_id"].isin(drop_ids)].copy()

print("Before:", len(pois2), "| After:", len(pois2_clean), "| Dropped:", len(pois2) - len(pois2_clean))

Before: 370 | After: 345 | Dropped: 25


In [79]:

latin_re = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ]")

mask_latin_name = pois2_clean["Name"].astype(str).str.contains(latin_re)
pois2_clean.loc[mask_latin_name, ["place_id","Name","City","retr_text_ru"]].head(30)
# print("Latin names:", int(mask_latin_name.sum()))

,place_id,Name,City,retr_text_ru
31,fallback:Владимир|Концертно-спортивный комплек...,Концертно-спортивный комплекс Art Hall,Владимир,Концертно-спортивный комплекс Art Hall. Город:...
41,fallback:Владимир|Музей хрусталя и стекла XVII...,Музей хрусталя и стекла XVIII-XXI веков,Владимир,Музей хрусталя и стекла XVIII-XXI веков. Город...
70,fallback:Владимир|Чернобыль - трагедия XX века...,Чернобыль - трагедия XX века,Владимир,Чернобыль - трагедия XX века. Город: Владимир....
117,wd:Q18331756,église arménienne de Vladimir,Владимир,église arménienne de Vladimir. Город: Владимир...
156,wd:Q28056804,Cerkiew św. Dymitra Sołuńskiego w Jarosławiu,Ярославль,Cerkiew św. Dymitra Sołuńskiego w Jarosławiu. ...
157,wd:Q28058985,Cerkiew Narodzenia Pańskiego w Jarosławiu,Ярославль,Cerkiew Narodzenia Pańskiego w Jarosławiu. Гор...
163,wd:Q30913943,Cerkiew Włodzimierskiej Ikony Matki Bożej w Ja...,Ярославль,Cerkiew Włodzimierskiej Ikony Matki Bożej w Ja...
203,wd:Q4165829,Домик Петра I,Нижний Новгород,Домик Петра I. Город: Нижний Новгород. Домик П...
228,wd:Q4306075,Музей «Литературная жизнь Урала XIX века»,Екатеринбург,Музей «Литературная жизнь Урала XIX века». Гор...
229,wd:Q4306077,Литературная жизнь Урала XX века,Екатеринбург,Литературная жизнь Урала XX века. Город: Екате...


In [80]:
view = pois2_clean[["place_id","Name","City","retr_text_ru"]].copy()
view["preview"] = view["retr_text_ru"].astype(str).str.replace(r"\s+", " ", regex=True).str.slice(0, 220)
view = view.sort_values(["City","Name"])
view[["place_id","Name","City","preview"]]

,place_id,Name,City,preview
0,fallback:Владимир|1941-1945|56.11119|40.36295,1941-1945,Владимир,1941-1945. Город: Владимир. Описание по фото: ...
1,fallback:Владимир|2000 летию Рождества Христов...,2000 летию Рождества Христова,Владимир,2000 летию Рождества Христова. Город: Владимир...
117,wd:Q18331756,église arménienne de Vladimir,Владимир,église arménienne de Vladimir. Город: Владимир...
346,wd:Q838264,Белокаменные памятники Владимира и Суздаля,Владимир,Белокаменные памятники Владимира и Суздаля. Го...
4,fallback:Владимир|Бобёр|56.12108|40.37801,Бобёр,Владимир,Бобёр. Город: Владимир. Описание по фото: бобё...
...,...,...,...,...
138,wd:Q2084804,церковь Иоанна Предтечи в Толчкове,Ярославль,церковь Иоанна Предтечи в Толчкове. Город: Яро...
94,wd:Q1365930,церковь Михаила Архангела,Ярославль,церковь Михаила Архангела. Город: Ярославль. П...
162,wd:Q30913942,церковь Николая Чудотворца,Ярославль,церковь Николая Чудотворца. Город: Ярославль. ...
110,wd:Q16715656,церковь Николы Рублёного,Ярославль,церковь Николы Рублёного. Город: Ярославль. хр...


In [81]:
# считаем буквы (только буквы, без цифр/пробелов/знаков)
re_cyr = re.compile(r"[А-Яа-яЁё]")
re_lat = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ]")

def count_re(pattern, s: str) -> int:
    return len(pattern.findall(str(s)))

def latin_ratio(name: str):
    name = str(name)
    cyr = count_re(re_cyr, name)
    lat = count_re(re_lat, name)
    letters = cyr + lat
    ratio = (lat / letters) if letters > 0 else 0.0
    return cyr, lat, letters, ratio

tmp_lang = pois2_clean.copy()
tmp_lang[["cyr_cnt","lat_cnt","letters_cnt","lat_ratio"]] = tmp_lang["Name"].apply(
    lambda x: pd.Series(latin_ratio(x))
)

# критерий "не русское имя":
# 1) кириллицы почти нет
# 2) латиницы достаточно много
# 3) доля латиницы высокая
mask_nonru = (tmp_lang["cyr_cnt"] <= 2) & (tmp_lang["lat_cnt"] >= 8) & (tmp_lang["lat_ratio"] >= 0.7)

print("Non-RU name candidates:", int(mask_nonru.sum()))
tmp_lang.loc[mask_nonru, ["place_id","Name","City","cyr_cnt","lat_cnt","lat_ratio","retr_text_ru"]].head(50)

Non-RU name candidates: 4


,place_id,Name,City,cyr_cnt,lat_cnt,lat_ratio,retr_text_ru
117,wd:Q18331756,église arménienne de Vladimir,Владимир,0.0,26.0,1.0,église arménienne de Vladimir. Город: Владимир...
156,wd:Q28056804,Cerkiew św. Dymitra Sołuńskiego w Jarosławiu,Ярославль,0.0,34.0,1.0,Cerkiew św. Dymitra Sołuńskiego w Jarosławiu. ...
157,wd:Q28058985,Cerkiew Narodzenia Pańskiego w Jarosławiu,Ярославль,0.0,35.0,1.0,Cerkiew Narodzenia Pańskiego w Jarosławiu. Гор...
163,wd:Q30913943,Cerkiew Włodzimierskiej Ikony Matki Bożej w Ja...,Ярославль,0.0,45.0,1.0,Cerkiew Włodzimierskiej Ikony Matki Bożej w Ja...


In [82]:
pois2_clean2 = pois2_clean.loc[~mask_nonru].reset_index(drop=True)
print("After non-RU-name drop:", len(pois2_clean), "->", len(pois2_clean2))

After non-RU-name drop: 345 -> 341


In [83]:
pois2_clean2

,place_id,WikiData,Name,City,Lat,Lon,description_best,ru_len,captions_top,n_rows,...,n_caps_good,n_caps_fb,captions_good,captions_fb,caps_en_used_list,caps_ru_used_list,caps_ru_used,retr_text_ru,rule_used,retr_len
0,fallback:Владимир|1941-1945|56.11119|40.36295,NaN,1941-1945,Владимир,56.111187,40.362949,,0,soldiers in a trench with a machine gun in fro...,30,...,2,0,soldiers in a trench with a machine gun in fro...,,[soldiers in a trench with a machine gun in fr...,"[солдат в траншее с пулеметом перед зданием, В...",солдат в траншее с пулеметом перед зданием | В...,1941-1945. Город: Владимир. Описание по фото: ...,NO_RU_USE_CAPS,143
1,fallback:Владимир|2000 летию Рождества Христов...,NaN,2000 летию Рождества Христова,Владимир,56.127674,40.408882,,0,cars are parked on the side of the road in fro...,58,...,5,0,monument in a snowy park with a statue in the ...,,[monument in a snowy park with a statue in the...,[памятник в снежном парке со статуей посередин...,памятник в снежном парке со статуей посередине...,2000 летию Рождества Христова. Город: Владимир...,NO_RU_USE_CAPS,327
2,fallback:Владимир|Бобёр|56.12108|40.37801,NaN,Бобёр,Владимир,56.121078,40.378010,,0,a beaver with an ax in his hand and a big axe ...,55,...,2,0,a beaver with an ax in his hand and a big axe ...,,[a beaver with an ax in his hand and a big axe...,[бобёр с топором в руке и большим топором в пр...,бобёр с топором в руке и большим топором в при...,Бобёр. Город: Владимир. Описание по фото: бобё...,NO_RU_USE_CAPS,164
3,fallback:Владимир|Великая Княгиня Ольга|56.129...,NaN,Великая Княгиня Ольга,Владимир,56.129597,40.392128,,0,a picture of a painting of a woman and a man i...,59,...,0,2,,a picture of a painting of a woman and a man i...,[a picture of a painting of a woman and a man ...,[фотографию картины женщины и мужчины в церкви...,фотографию картины женщины и мужчины в церкви ...,Великая Княгиня Ольга. Город: Владимир. Описан...,NO_RU_USE_CAPS,151
4,fallback:Владимир|Вечный огонь|56.12003|40.364,NaN,Вечный огонь,Владимир,56.120033,40.364002,,0,arafed monument in the center of a park with a...,31,...,5,0,view of a large white building with a clock to...,,[view of a large white building with a clock t...,"[вид на большое белое здание с часовой башней,...",вид на большое белое здание с часовой башней |...,Вечный огонь. Город: Владимир. Описание по фот...,NO_RU_USE_CAPS,298
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
336,wd:Q97678623,Q97678623,Церковь в честь Рождества Богородицы,Нижний Новгород,56.209595,43.755871,Церковь Рождества Богородицы в селе Гнилицы,43,snowy night scene of a church with a clock tow...,57,...,5,0,view of a church with a clock tower and a rive...,,[view of a church with a clock tower and a riv...,[Вид на церковь с часовой башней и рекой на за...,Вид на церковь с часовой башней и рекой на зад...,Церковь в честь Рождества Богородицы. Город: Н...,RU_SHORT_PLUS_CAPS,418
337,wd:Q97999957,Q97999957,Здание Удельной конторы,Нижний Новгород,56.325153,44.021175,Здание Удельной конторы,23,cars are parked on the street in front of a la...,28,...,5,0,black and white photograph of a building with ...,,[black and white photograph of a building with...,[Черная и белая фотография здания с часами на ...,Черная и белая фотография здания с часами на н...,Здание Удельной конторы. Город: Нижний Новгоро...,RU_SHORT_PLUS_CAPS,348
338,wd:Q98381215,Q98381215,Дом архитектора,Нижний Новгород,56.329483,44.009949,Жилой дом горьковского архитектора А. А. Яковлева,49,cars are parked in front of a large white buil...,27,...,0,2,,cars parked in front of a pink building with a...,[cars parked in front of a pink building with ...,"[Автомобили, припаркованные перед розовым здан...","Автомобили, припаркованные перед розовым здани...",Дом архитектора. Город: Нижний Новгород. Жилой...,RU_SHORT_PLUS_CAPS,233
339,wd:Q99622643,Q99622643,Дворец вице-губернатора,Нижний Новгород,56.327217,44.003147,"административное здание в Нижнем Новгороде,

In [84]:
pois_final = pois2_clean2.copy()

# базовые проверки
print("rows:", len(pois_final))
print("unique place_id:", pois_final["place_id"].nunique())
print("empty retr_text_ru:", int((pois_final["retr_text_ru"].astype(str).str.strip() == "").sum()))

# длины
pois_final["retr_len"] = pois_final["retr_text_ru"].astype(str).str.len()
print(pois_final["retr_len"].describe())

# если хочешь — выкинем совсем пустые (на всякий)
pois_final = pois_final[pois_final["retr_len"] > 0].reset_index(drop=True)

# сохранить
pois_final.to_csv("pois_final_341.csv", index=False, encoding="utf-8")
print("saved: pois_final_341.csv")

rows: 341
unique place_id: 341
empty retr_text_ru: 0
count    341.000000
mean     321.826979
std       79.539276
min       99.000000
25%      297.000000
50%      341.000000
75%      373.000000
max      492.000000
Name: retr_len, dtype: float64
saved: pois_final_341.csv


## 5) Baseline retrieval (BM25 + FAISS/ Dense / Hybrid)





**Почему без чанкинга:** каждый `POI` — короткий документ (сотни символов), дробить на чанки нет смысла: потеряем целостность “1 объект = 1 документ”, усложним индексацию и оценку.

**Почему BM25 baseline:** сильный, быстрый, понятный, хорошо ловит точные совпадения по сущностям (названия, города).

Зачем `dense+FAISS`: улучшает качество на перефразированиях и “разговорных” запросах; `FAISS` даёт векторный индекс как стандартный компонент.

## Шаг 1. Baseline retrieval: BM25 (лексический поиск)

**BM25** — классический и очень сильный baseline для retrieval.  
Он работает как улучшенный "мешок слов": учитывает частоту слов в документе и то, насколько слово редкое по корпусу.

Почему начинаем с BM25:
- стабильно работает на небольших коллекциях (у нас 341 документ),
- хорошо ловит точные совпадения (названия, города, тип объекта),
- быстро и прозрачно.

На выходе мы получаем функцию `bm25_search(query, top_k)`, которая возвращает top-k POI по запросу.

In [85]:
# BM25 baseline

from rank_bm25 import BM25Okapi
# (1) токенизация RU: простая, но адекватная для baseline
token_re = re.compile(r"[А-Яа-яЁёA-Za-z0-9]+")

def tokenize_ru(text: str):
    return token_re.findall(str(text).lower())

docs = pois_final["retr_text_ru"].fillna("").astype(str).tolist()
tokenized_docs = [tokenize_ru(t) for t in docs]

bm25 = BM25Okapi(tokenized_docs)

def bm25_search(query: str, top_k: int = 5):
    q_tokens = tokenize_ru(query)
    scores = bm25.get_scores(q_tokens)
    idx = np.argsort(scores)[::-1][:top_k]
    out = pois_final.loc[idx, ["place_id","Name","City","retr_text_ru"]].copy()
    out["bm25_score"] = scores[idx]
    return out

# smoke-test
tests = [
    "музей ложки владимир",
    "театр ярославль",
    "монастырь ярославль",
    "кремль нижний новгород",
]
for q in tests:
    print("\nQUERY:", q)
    display(bm25_search(q, top_k=5)[["bm25_score","Name","City"]])


QUERY: музей ложки владимир


,bm25_score,Name,City
307,13.840249,Музей ложки,Владимир
34,4.562383,"Музей науки и человека ЭВРИКА, г. Владимир",Владимир
73,4.250083,Дом-музей Столетовых,Владимир
35,4.202262,Музей непридуманных историй,Владимир
131,3.707814,Исторический музей,Владимир



QUERY: театр ярославль


,bm25_score,Name,City
275,6.482333,Ярославский государственный театр юного зрител...,Ярославль
277,6.482333,Ярославский камерный театр,Ярославль
274,5.858389,Театр кукол,Ярославль
107,5.710285,Театр оперы и балета,Нижний Новгород
241,5.106380,Российский государственный академический театр...,Ярославль



QUERY: монастырь ярославль


,bm25_score,Name,City
238,6.459251,Петровский монастырь (Ярославль),Ярославль
195,6.219455,Казанский женский монастырь,Ярославль
221,5.527601,Николо-Сковородский монастырь,Ярославль
196,5.453461,Кирило-Афанасиевский мужской монастырь,Ярославль
79,5.242708,Спасо-Преображенский монастырь,Ярославль



QUERY: кремль нижний новгород


,bm25_score,Name,City
95,6.175313,Нижегородский кремль,Нижний Новгород
67,6.122959,Нижегородский Кремль,Нижний Новгород
306,5.508929,Владимирская митрополия,Владимир
183,3.453370,Дом культуры ГУВД (Нижний Новгород),Нижний Новгород
340,3.272372,Дом Сироткина,Нижний Новгород


## Шаг 2. Dense retrieval: эмбеддинги (семантический поиск)

Dense retrieval ищет документы по **смыслу**, а не по точным словам:
- лучше ловит перефразирования,
- может находить релевантные POI даже при "разговорных" запросах.

Мы создаём эмбеддинг для каждого `retr_text_ru` и будем считать similarity между запросом и документами.
Пока без FAISS: просто через матричное умножение (быстро при 341 документе).

In [86]:
# Dense retrieval: embeddings
from sentence_transformers import SentenceTransformer

# Модель лучше взять multilingual, чтобы RU работал стабильно
# Варианты:
# - "intfloat/multilingual-e5-base"
# - "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
EMB_MODEL = "intfloat/multilingual-e5-base"

embedder = SentenceTransformer(EMB_MODEL)

# e5 желательно с префиксами "query: " / "passage: "
def make_passage(text):
    return "passage: " + str(text)

def make_query(text):
    return "query: " + str(text)

passages = [make_passage(t) for t in pois_final["retr_text_ru"].fillna("").astype(str).tolist()]
doc_emb = embedder.encode(passages, normalize_embeddings=True, batch_size=32, show_progress_bar=True)

def dense_search(query: str, top_k: int = 5):
    q_emb = embedder.encode([make_query(query)], normalize_embeddings=True)[0]
    scores = doc_emb @ q_emb  # cosine similarity, т.к. нормализовано
    idx = np.argsort(scores)[::-1][:top_k]
    out = pois_final.loc[idx, ["place_id","Name","City","retr_text_ru"]].copy()
    out["dense_score"] = scores[idx]
    return out

# smoke-test
for q in tests:
    print("\nQUERY:", q)
    display(dense_search(q, top_k=5)[["dense_score","Name","City"]])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/11 [00:00<?, ?it/s]


QUERY: музей ложки владимир


,dense_score,Name,City
307,0.883228,Музей ложки,Владимир
131,0.826850,Исторический музей,Владимир
163,0.821199,Владимиро-Суздальский музей-заповедник (центра...,Владимир
35,0.819343,Музей непридуманных историй,Владимир
41,0.818289,Огни Владимира,Владимир


...

,dense_score,Name,City
275,0.867499,Ярославский государственный театр юного зрител...,Ярославль
274,0.861813,Театр кукол,Ярославль
277,0.852618,Ярославский камерный театр,Ярославль
241,0.841948,Российский государственный академический театр...,Ярославль
215,0.836300,Музыка и время,Ярославль



QUERY: монастырь ярославль


,dense_score,Name,City
238,0.875948,Петровский монастырь (Ярославль),Ярославль
195,0.864706,Казанский женский монастырь,Ярославль
196,0.859062,Кирило-Афанасиевский мужской монастырь,Ярославль
79,0.855525,Спасо-Преображенский монастырь,Ярославль
221,0.849188,Николо-Сковородский монастырь,Ярославль



QUERY: кремль нижний новгород


,dense_score,Name,City
95,0.862901,Нижегородский кремль,Нижний Новгород
285,0.858632,Северная башня,Нижний Новгород
67,0.857578,Нижегородский Кремль,Нижний Новгород
286,0.843090,Кладовая башня,Нижний Новгород
139,0.841719,Георгиевская башня,Нижний Новгород


## Шаг 3. Dense retrieval через FAISS (векторный индекс)

При 341 документах можно искать и без FAISS, но:
- FAISS — стандартный инструмент для векторного поиска,
- пригодится, если будем расширишь базу до тысяч/миллионов документов.

Мы строим FAISS-индекс по эмбеддингам документов и делаем поиск top-k.

In [87]:
# FAISS index

import faiss

# doc_emb уже нормализован (cosine ~ dot product)
doc_emb_np = np.asarray(doc_emb, dtype="float32")
dim = doc_emb_np.shape[1]

index = faiss.IndexFlatIP(dim)  # Inner Product для cosine при нормализации
index.add(doc_emb_np)

def faiss_dense_search(query: str, top_k: int = 5):
    q_emb = embedder.encode([make_query(query)], normalize_embeddings=True)
    q_emb = np.asarray(q_emb, dtype="float32")
    scores, idx = index.search(q_emb, top_k)  # scores: (1,k), idx: (1,k)
    idx = idx[0]
    scores = scores[0]
    out = pois_final.loc[idx, ["place_id","Name","City","retr_text_ru"]].copy()
    out["faiss_score"] = scores
    return out

# smoke-test
for q in tests:
    print("\nQUERY:", q)
    display(faiss_dense_search(q, top_k=5)[["faiss_score","Name","City"]])


QUERY: музей ложки владимир


,faiss_score,Name,City
307,0.883228,Музей ложки,Владимир
131,0.826850,Исторический музей,Владимир
163,0.821199,Владимиро-Суздальский музей-заповедник (центра...,Владимир
35,0.819343,Музей непридуманных историй,Владимир
41,0.818289,Огни Владимира,Владимир



QUERY: театр ярославль


,faiss_score,Name,City
275,0.867499,Ярославский государственный театр юного зрител...,Ярославль
274,0.861813,Театр кукол,Ярославль
277,0.852617,Ярославский камерный театр,Ярославль
241,0.841948,Российский государственный академический театр...,Ярославль
215,0.836300,Музыка и время,Ярославль



QUERY: монастырь ярославль


,faiss_score,Name,City
238,0.875948,Петровский монастырь (Ярославль),Ярославль
195,0.864706,Казанский женский монастырь,Ярославль
196,0.859062,Кирило-Афанасиевский мужской монастырь,Ярославль
79,0.855525,Спасо-Преображенский монастырь,Ярославль
221,0.849188,Николо-Сковородский монастырь,Ярославль



QUERY: кремль нижний новгород


,faiss_score,Name,City
95,0.862901,Нижегородский кремль,Нижний Новгород
285,0.858632,Северная башня,Нижний Новгород
67,0.857578,Нижегородский Кремль,Нижний Новгород
286,0.843090,Кладовая башня,Нижний Новгород
139,0.841719,Георгиевская башня,Нижний Новгород


## Eval (быстрый baseline): self-retrieval

Чтобы сравнить BM25 vs Dense vs FAISS, нам нужна общая метрика качества.
Сырые scores сравнивать нельзя — они в разных шкалах.

Быстрый тест без ручной разметки:
- для каждого POI делаем запрос `Name + City`
- проверяем, попал ли *тот же* `place_id` в top-k выдачи

Метрики:
- **Recall@k**: доля запросов, где нужный `place_id` найден в top-k
- **MRR@k**: учитывает позицию (чем ближе к топ-1, тем лучше)

In [88]:
def recall_mrr_at_k(retriever_fn, k=5):
    hits = 0
    rr_sum = 0.0

    for i, row in pois_final.iterrows():
        q = f"{row['Name']} {row['City']}"
        res = retriever_fn(q, top_k=k)

        # ожидаем, что res содержит place_id
        ids = res["place_id"].tolist()
        target = row["place_id"]

        if target in ids:
            hits += 1
            rank = ids.index(target) + 1
            rr_sum += 1.0 / rank

    n = len(pois_final)
    recall = hits / n
    mrr = rr_sum / n
    return recall, mrr

def bm25_fn(q, top_k=5):
    return bm25_search(q, top_k=top_k)

def dense_fn(q, top_k=5):
    return dense_search(q, top_k=top_k)

def faiss_fn(q, top_k=5):
    return faiss_dense_search(q, top_k=top_k)

for k in [1,3,5,10]:
    r_b, m_b = recall_mrr_at_k(bm25_fn, k=k)
    r_d, m_d = recall_mrr_at_k(dense_fn, k=k)
    r_f, m_f = recall_mrr_at_k(faiss_fn, k=k)
    print(f"\n@{k}")
    print(f"BM25 : recall={r_b:.3f}, mrr={m_b:.3f}")
    print(f"Dense: recall={r_d:.3f}, mrr={m_d:.3f}")
    print(f"FAISS: recall={r_f:.3f}, mrr={m_f:.3f}")


@1
BM25 : recall=0.947, mrr=0.947
Dense: recall=0.977, mrr=0.977
FAISS: recall=0.977, mrr=0.977

@3
BM25 : recall=0.991, mrr=0.969
Dense: recall=0.994, mrr=0.984
FAISS: recall=0.994, mrr=0.984

@5
BM25 : recall=0.991, mrr=0.969
Dense: recall=1.000, mrr=0.986
FAISS: recall=1.000, mrr=0.986

@10
BM25 : recall=0.994, mrr=0.969
Dense: recall=1.000, mrr=0.986
FAISS: recall=1.000, mrr=0.986


Чего мы ожидали от этих метрик

От self-retrieval мы ожидали:

не нули (значит всё работает)

примерно BM25 ≈ dense

dense иногда чуть лучше (он “прощает” мелкие вариации, особенно если в Name есть странности)

## Human-eval (15–20 запросов)

Self-retrieval даёт слишком оптимистичные метрики, потому что запрос = "Name + City".
Чтобы честно сравнить BM25 vs Dense (и потом Hybrid), делаем небольшой набор "человеческих" запросов.

Процедура:
1) Берём 15–20 запросов: часть по городу, часть по типу объекта (музей/храм/театр), часть разговорных.
2) Для каждого запроса фиксируем `target_place_id` (какой POI считаем правильным).
3) Считаем Recall@k и MRR@k для разных retriever'ов.

Это будет основная сравнительная метрика для выбора подхода retrieval.

In [89]:
import random
random.seed(42)

df = pois_final.copy()

# грубо выделяем "тип" из названия (очень приблизительно, но помогает сделать запросы)
TYPE_WORDS = ["музей","храм","собор","церковь","монастырь","театр","памятник","усадьба","башня","мост","парк","кремль"]

def guess_type(name: str):
    n = str(name).lower()
    for w in TYPE_WORDS:
        if w in n:
            return w
    return None

df["type_guess"] = df["Name"].apply(guess_type)

# выбираем несколько POI разных типов
samples = []
for w in ["музей","храм","театр","монастырь","усадьба","памятник","мост","парк","кремль"]:
    sub = df[df["type_guess"] == w]
    if len(sub) > 0:
        samples.append(sub.sample(1, random_state=42).iloc[0])

# если мало набралось — добиваем рандомом
while len(samples) < 10:
    samples.append(df.sample(1, random_state=42+len(samples)).iloc[0])

# строим кандидатные запросы
cands = []
for r in samples:
    name, city, pid, typ = r["Name"], r["City"], r["place_id"], r["type_guess"]
    if typ:
        cands += [
            (f"{typ} {city}", pid),
            (f"расскажи про {name}", pid),
            (f"что это за {name} в городе {city}?", pid),
        ]
    else:
        cands += [
            (f"расскажи про {name}", pid),
            (f"что посмотреть в {city}?", pid),
        ]

# добавим несколько "разговорных" общих
cities = df["City"].dropna().astype(str).unique().tolist()
for city in random.sample(cities, min(5, len(cities))):
    cands.append((f"куда сходить туристу в {city}?", None))
    cands.append((f"интересные места в {city}", None))

# оставим первые 20 уникальных запросов
seen = set()
rows = []
for q, pid in cands:
    if q.lower() in seen:
        continue
    seen.add(q.lower())
    rows.append({"query": q, "target_place_id": pid})
    if len(rows) >= 20:
        break

eval_df = pd.DataFrame(rows)
eval_df

,query,target_place_id
0,музей Ярославль,wd:Q4538889
1,расскажи про Ярославский музей-заповедник,wd:Q4538889
2,что это за Ярославский музей-заповедник в горо...,wd:Q4538889
3,храм Екатеринбург,wd:Q4411412
4,расскажи про Свято-Никольский храм (Екатеринбург),wd:Q4411412
5,что это за Свято-Никольский храм (Екатеринбург...,wd:Q4411412
6,театр Ярославль,wd:Q4538885
7,расскажи про Ярославский камерный театр,wd:Q4538885
8,что это за Ярославский камерный театр в городе...,wd:Q4538885
9,монастырь Ярославль,wd:Q4221355


In [90]:
def eval_retriever(eval_df: pd.DataFrame, retriever_fn, k_list=(1,3,5,10)):
    # оставляем только строки с target
    edf = eval_df.dropna(subset=["target_place_id"]).copy()
    edf["target_place_id"] = edf["target_place_id"].astype(str)

    results = []
    for k in k_list:
        hits = 0
        rr_sum = 0.0
        for _, row in edf.iterrows():
            q = row["query"]
            target = row["target_place_id"]
            res = retriever_fn(q, top_k=k)
            ids = res["place_id"].astype(str).tolist()

            if target in ids:
                hits += 1
                rank = ids.index(target) + 1
                rr_sum += 1.0 / rank

        n = len(edf)
        results.append({
            "k": k,
            "n_queries": n,
            "recall@k": hits / n if n else 0.0,
            "mrr@k": rr_sum / n if n else 0.0,
        })
    return pd.DataFrame(results)

# wrappers
def bm25_fn(q, top_k=5):
    return bm25_search(q, top_k=top_k)

def dense_fn(q, top_k=5):
    return faiss_dense_search(q, top_k=top_k)

bm25_metrics = eval_retriever(eval_df, bm25_fn)
dense_metrics = eval_retriever(eval_df, dense_fn)

print("BM25 metrics:")
display(bm25_metrics)

print("Dense/FAISS metrics:")
display(dense_metrics)

BM25 metrics:


,k,n_queries,recall@k,mrr@k
0,1,20,0.70,0.700000
1,3,20,0.90,0.791667
2,5,20,0.95,0.804167
3,10,20,1.00,0.812500


Dense/FAISS metrics:


,k,n_queries,recall@k,mrr@k
0,1,20,0.70,0.700000
1,3,20,0.95,0.816667
2,5,20,1.00,0.829167
3,10,20,1.00,0.829167


MRR@k (Mean Reciprocal Rank)

MRR учитывает насколько высоко стоит правильный документ.

Для одного запроса:

если правильный doc на месте 1 → вклад = 1/1 = 1.0

на месте 2 → вклад = 1/2 = 0.5

на месте 3 → вклад = 1/3 ≈ 0.333

…
Если правильного нет в топ-k → вклад = 0.

mrr@k = среднее этих вкладов по всем запросам.

MRR полезнее recall, потому что:

recall говорит “попал / не попал”

mrr говорит “насколько близко к топу”, что критично для RAG (обычно в prompt реально берут 3–5 документов).

Recall@k

Для каждого запроса есть “правильный” документ (gt doc_id).

recall@k = доля запросов, где правильный doc попал в топ-k.

Пример: recall@1 = 0.70 значит:

для 20 запросов

в 14 случаях из 20 (70%) правильный doc оказался на 1 месте.

recall@10 = 1.00 значит:

для всех 20 запросов правильный doc был где-то в топ-10.

Что значит k = 1, 3, 5, 10

k — это “сколько документов мы возвращаем в топе”.

k=1: мы берём только самый первый документ (top-1).

k=3: берём топ-3.

k=5: берём топ-5.

k=10: берём топ-10.

Почему именно такие? Это стандартные “контрольные точки”: top-1 (самое строгое), top-3/5 (похоже на то, что реально пойдёт в prompt), top-10 (верхняя граница, где recall обычно уже почти максимум).

In [91]:
TOP_K = 7

city_intent_queries = [
    {"query": "куда сходить в Ярославле с детьми", "city": "Ярославль"},
    {"query": "чем заняться во Владимире в выходные", "city": "Владимир"},
    {"query": "культурные мероприятия Владимир", "city": "Владимир"},
    {"query": "лучшие музеи в Нижнем Новгороде", "city": "Нижний Новгород"},
    {"query": "куда сходить вечером в Нижнем Новгороде", "city": "Нижний Новгород"},
    {"query": "что посмотреть в Екатеринбурге за 1 день", "city": "Екатеринбург"},
    {"query": "парки и прогулочные места в Ярославле", "city": "Яросавль"},  # опечатка специально
]

# --- 0) подготовим индексы ---
pois_by_id = pois_final.set_index("place_id", drop=False)

# простая нормализация названий городов (опечатки/варианты)
CITY_ALIASES = {
    "Яросавль": "Ярославль",
    "Нижний": "Нижний Новгород",
    "ЕКБ": "Екатеринбург",
}

def norm_city(city: str) -> str:
    c = str(city).strip()
    return CITY_ALIASES.get(c, c)

def hits_to_df(hits: object) -> pd.DataFrame:
    """Приводим результат поиска к DataFrame с колонками place_id и score."""
    if isinstance(hits, pd.DataFrame):
        # найдём колонку score
        score_col = None
        for c in ["bm25_score", "dense_score", "score"]:
            if c in hits.columns:
                score_col = c
                break
        if score_col is None:
            out = hits[["place_id"]].copy()
            out["score"] = None
        else:
            out = hits[["place_id", score_col]].copy().rename(columns={score_col: "score"})
        return out

    # list of tuples / dicts / ids
    rows = []
    for h in list(hits):
        if isinstance(h, dict):
            rows.append({"place_id": h.get("place_id"), "score": h.get("score")})
        elif isinstance(h, (list, tuple)):
            pid = h[0] if len(h) > 0 else None
            sc  = h[1] if len(h) > 1 else None
            rows.append({"place_id": pid, "score": sc})
        else:
            rows.append({"place_id": h, "score": None})
    return pd.DataFrame(rows)

def pretty_df(df_hits: pd.DataFrame, title: str, top_k: int = TOP_K):
    lines = [title]
    for i, r in df_hits.head(top_k).iterrows():
        pid = r["place_id"]
        score = r.get("score", None)
        if pid not in pois_by_id.index:
            lines.append(f"{len(lines):02d}. <missing> place_id={pid} | score={score}")
            continue
        row = pois_by_id.loc[pid]
        name = str(row.get("Name", ""))
        city = str(row.get("City", ""))
        txt = str(row.get("retr_text_ru", ""))
        preview = (txt[:220] + "…") if len(txt) > 220 else txt
        score_str = f"{score:.4f}" if isinstance(score, (int, float)) else str(score)
        lines.append(f"{len(lines):02d}. {name} — {city} | score={score_str}\n    ↳ {preview}")
    return "\n".join(lines)

# --- 1) Smoke test: city-intent queries ---
for ex in city_intent_queries:
    q = ex["query"]
    city = norm_city(ex["city"])

    print("=" * 120)
    print(f"QUERY: {q}")
    print(f"CITY:  {city}\n")

    # BM25
    if "bm25_search" in globals():
        bm25_hits = hits_to_df(bm25_search(q, top_k=TOP_K))
        # city filter
        bm25_hits = bm25_hits.merge(pois_final[["place_id", "City"]], on="place_id", how="left")
        bm25_hits = bm25_hits[bm25_hits["City"] == city].drop(columns=["City"])
        print(pretty_df(bm25_hits, "BM25 top hits (city-filtered):", top_k=TOP_K))
        print()
    else:
        print("BM25 not available: bm25_search() is not defined.\n")

    # Dense/FAISS
    if "dense_search" in globals():
        dense_hits = hits_to_df(dense_search(q, top_k=TOP_K))
        dense_hits = dense_hits.merge(pois_final[["place_id", "City"]], on="place_id", how="left")
        dense_hits = dense_hits[dense_hits["City"] == city].drop(columns=["City"])
        print(pretty_df(dense_hits, "Dense/FAISS top hits (city-filtered):", top_k=TOP_K))
        print()
    else:
        print("Dense/FAISS not available: dense_search() is not defined.\n")

QUERY: куда сходить в Ярославле с детьми
CITY:  Ярославль

BM25 top hits (city-filtered):
01. Ярославский государственный театр юного зрителя им. В. С. Розова — Ярославль | score=6.7501
    ↳ Ярославский государственный театр юного зрителя им. В. С. Розова. Город: Ярославль. Театр в Ярославле Описание по фото: Там большое здание с часовой башней на заднем плане. | фонтан перед зданием с фонтаном посередине
02. Ф. Г. Волкову — Ярославль | score=6.5283
    ↳ Ф. Г. Волкову. Город: Ярославль. Памятник в Ярославле Описание по фото: фото здания с цветочным садом перед ним | Статуя человека, держащего бейсбольную биту в парке | Статуя человека с бейсбольной битой в парке | Статуя…
03. Часовня Александра Невского — Ярославль | score=6.4807
    ↳ Часовня Александра Невского. Город: Ярославль. часовня в Ярославле Описание по фото: здание с часовой башней в парке с голубым небом | люди ходят вокруг площади перед церковью с часовой башней | изображение коллажа разли…
04. Успенский собор — Ярославл

In [92]:
# ---- 1) Быстрые проверки таблицы ----
print("len(pois_final):", len(pois_final))
print("place_id unique:", pois_final["place_id"].nunique())
dup_place = pois_final["place_id"].duplicated().sum()
print("duplicated place_id rows:", int(dup_place))

# одинаковые retr_text_ru (полезно знать)
if "retr_text_ru" in pois_final.columns:
    dup_text = pois_final["retr_text_ru"].astype(str).duplicated().sum()
    print("duplicated retr_text_ru rows:", int(dup_text))

# ---- 2) Проверка выдачи Dense/FAISS на конкретном запросе ----
q = "чем заняться во Владимире в выходные"
TOP_K = 10

dense_hits = dense_search(q, top_k=TOP_K)

# Попробуем вытащить doc_idx/doc_id как число (индекс в pois_final)
def _extract_idx(hit):
    # возможные форматы:
    # (idx, score)
    # {"doc_id": idx, "score": s}
    # {"idx": idx, "score": s}
    if isinstance(hit, (tuple, list)) and len(hit) >= 1:
        return int(hit[0])
    if isinstance(hit, dict):
        for k in ["doc_id", "idx", "id", "row_id"]:
            if k in hit:
                return int(hit[k])
    return None

idxs = [ _extract_idx(h) for h in dense_hits ]
print("\nDense hits raw idxs:", idxs)

# если idx None -> значит dense_search возвращает place_id, а не индексы — тоже ок, просто нужно другое сопоставление
n_none = sum(i is None for i in idxs)
if n_none:
    print(f"WARNING: {n_none}/{len(idxs)} hits have no numeric idx. dense_search format may be place_id-based.")

# ---- 3) Проверка повторов индексов ----
idxs_valid = [i for i in idxs if i is not None]
if idxs_valid:
    vc = pd.Series(idxs_valid).value_counts()
    reps = vc[vc > 1]
    print("\nRepeated idxs in top-k:", reps.to_dict() if len(reps) else "none")

    # покажем конкретные повторяющиеся строки
    if len(reps):
        rep_idx = reps.index.tolist()[:5]
        print("\nRepeated rows preview:")
        display(pois_final.iloc[rep_idx][["place_id", "Name", "City", "retr_text_ru"]].head(10))
else:
    print("\nNo numeric idxs to analyze repeats; dense_search might already return place_id.")

len(pois_final): 341
place_id unique: 341
duplicated place_id rows: 0
duplicated retr_text_ru rows: 3

Dense hits raw idxs: [None, None, None, None, None]

No numeric idxs to analyze repeats; dense_search might already return place_id.


### 5.1) Гибридный поиск

Собираем **late-fusion retriever**, который объединяет:
- BM25 (лексическое совпадение слов),
- dense / FAISS retrieval (семантическая близость).

Идея простая: для каждого запроса получаем кандидатов из обеих веток, нормируем score и смешиваем их с весом `alpha`. Такой retriever затем используем как основной источник контекста для генерации.

In [93]:
import numpy as np

def _to_hits_list(x):
    """Приводим вывод bm25_search/dense_search к списку хитов."""
    if x is None:
        return []
    if isinstance(x, (list, tuple)):
        return list(x)
    # если вдруг приходит numpy array
    try:
        return list(x)
    except Exception:
        return [x]

def _hit_doc_id_and_score(hit):
    """
    Возвращает (doc_id, score) где doc_id — это индекс (int) строки в pois_final.
    Поддерживаем несколько форматов:
      - (doc_id, score)
      - {"doc_id": ..., "score": ...}
      - {"idx": ..., "score": ...}
    """
    if isinstance(hit, (tuple, list)):
        if len(hit) >= 2:
            return int(hit[0]), float(hit[1])
        if len(hit) == 1:
            return int(hit[0]), 0.0

    if isinstance(hit, dict):
        doc_id = None
        for k in ["doc_id", "idx", "id", "row_id"]:
            if k in hit:
                doc_id = int(hit[k])
                break
        score = float(hit.get("score", hit.get("bm25_score", hit.get("dense_score", 0.0))))
        if doc_id is None:
            raise ValueError(f"Can't parse doc_id from hit dict: {hit}")
        return doc_id, score

    # если прилетело просто число
    if isinstance(hit, (int, np.integer)):
        return int(hit), 0.0

    raise ValueError(f"Unknown hit format: {type(hit)} -> {hit}")

def _minmax_norm(scores: dict):
    """
    Min-Max нормализация словаря {doc_id: score} в [0,1].
    Если все одинаковые -> возвращаем 0.0 для всех (или 1.0 — неважно, главное стабильность).
    """
    if not scores:
        return {}
    vals = np.array(list(scores.values()), dtype=float)
    lo, hi = float(vals.min()), float(vals.max())
    if hi - lo < 1e-12:
        return {k: 0.0 for k in scores.keys()}
    return {k: (float(v) - lo) / (hi - lo) for k, v in scores.items()}

In [94]:
# Hybrid retrieval (late fusion)
# score = alpha * bm25_norm + (1-alpha) * dense_norm

def _minmax_norm_np(scores) -> np.ndarray:
    """Robust min-max for list/np/Series -> np.ndarray in [0,1]."""
    scores = np.asarray(scores, dtype=float)
    if scores.size == 0:
        return scores
    # если вдруг есть nan
    if np.isnan(scores).all():
        return np.zeros_like(scores, dtype=float)

    smin = float(np.nanmin(scores))
    smax = float(np.nanmax(scores))
    if (not np.isfinite(smin)) or (not np.isfinite(smax)) or (smax - smin < 1e-12):
        # все одинаковые/плохие -> единицы, чтобы не убить сигнал
        return np.ones_like(scores, dtype=float)
    return (scores - smin) / (smax - smin)

def hybrid_search(q: str, top_k: int = 10, alpha: float = 0.5, pool_k: int | None = None) -> pd.DataFrame:
    """
    Возвращает DataFrame: place_id, score
    - pool_k: сколько взять кандидатов из каждого ретривера ДО слияния (по умолчанию max(5*top_k, 20))
    """
    if pool_k is None:
        pool_k = max(top_k * 5, 20)

    # 1) кандидаты
    bm25_df = hits_to_df(bm25_search(q, top_k=pool_k))
    dense_df = hits_to_df(faiss_dense_search(q, top_k=pool_k))

    # 2) чистим
    bm25_df = bm25_df.dropna(subset=["place_id"]).copy()
    dense_df = dense_df.dropna(subset=["place_id"]).copy()

    # score None/str -> float
    bm25_df["score"] = pd.to_numeric(bm25_df.get("score", 0.0), errors="coerce").fillna(0.0)
    dense_df["score"] = pd.to_numeric(dense_df.get("score", 0.0), errors="coerce").fillna(0.0)

    # 3) нормализация
    bm25_df["bm25_n"] = _minmax_norm_np(bm25_df["score"].to_numpy())
    dense_df["dense_n"] = _minmax_norm_np(dense_df["score"].to_numpy())

    bm25_df = bm25_df[["place_id", "bm25_n"]]
    dense_df = dense_df[["place_id", "dense_n"]]

    # 4) union + fill 0
    merged = bm25_df.merge(dense_df, on="place_id", how="outer").fillna(0.0)

    # 5) фьюз (ВАЖНО: alpha у тебя сейчас весит BM25, (1-alpha) — dense)
    merged["score"] = alpha * merged["bm25_n"] + (1.0 - alpha) * merged["dense_n"]

    merged = merged.drop_duplicates(subset=["place_id"], keep="first")
    merged = merged.sort_values("score", ascending=False).head(top_k).reset_index(drop=True)
    return merged[["place_id", "score"]]

# ---- wrappers под eval_retriever ----
def hybrid_fn_factory(alpha: float):
    def _fn(q, top_k=5):
        return hybrid_search(q, top_k=top_k, alpha=alpha)
    return _fn

# ---- метрики как у BM25/Dense ----
alphas = [0.2, 0.5, 0.8]
hyb_all = []
for a in alphas:
    hyb_metrics = eval_retriever(eval_df, hybrid_fn_factory(a))
    hyb_metrics = hyb_metrics.copy()
    hyb_metrics["alpha"] = a
    hyb_all.append(hyb_metrics)

hyb_metrics_df = pd.concat(hyb_all, ignore_index=True)
print("Hybrid metrics:")
display(hyb_metrics_df)

Hybrid metrics:


,k,n_queries,recall@k,mrr@k,alpha
0,1,20,0.75,0.750000,0.2
1,3,20,0.90,0.816667,0.2
2,5,20,0.95,0.829167,0.2
3,10,20,1.00,0.862500,0.2
4,1,20,0.75,0.750000,0.5
5,3,20,0.90,0.816667,0.5
6,5,20,0.95,0.829167,0.5
7,10,20,1.00,0.862500,0.5
8,1,20,0.75,0.750000,0.8
9,3,20,0.90,0.816667,0.8


In [95]:
TOP_K = 7
ALPHA = 0.3

for ex in city_intent_queries:
    q = ex["query"]
    city = norm_city(ex["city"])

    print("=" * 120)
    print(f"QUERY: {q}")
    print(f"CITY:  {city}\n")

    # Hybrid
    hyb_hits = hybrid_search(q, top_k=TOP_K, alpha=ALPHA)

    # city filter
    hyb_hits = hyb_hits.merge(pois_final[["place_id", "City"]], on="place_id", how="left")
    hyb_hits = hyb_hits[hyb_hits["City"] == city].drop(columns=["City"])

    print(pretty_df(hyb_hits, f"Hybrid top hits (alpha={ALPHA}, city-filtered):", top_k=TOP_K))
    print()

QUERY: куда сходить в Ярославле с детьми
CITY:  Ярославль

Hybrid top hits (alpha=0.3, city-filtered):
01. Ярославский государственный театр юного зрителя им. В. С. Розова — Ярославль | score=1.0000
    ↳ Ярославский государственный театр юного зрителя им. В. С. Розова. Город: Ярославль. Театр в Ярославле Описание по фото: Там большое здание с часовой башней на заднем плане. | фонтан перед зданием с фонтаном посередине
02. Часовня Александра Невского — Ярославль | score=0.9641
    ↳ Часовня Александра Невского. Город: Ярославль. часовня в Ярославле Описание по фото: здание с часовой башней в парке с голубым небом | люди ходят вокруг площади перед церковью с часовой башней | изображение коллажа разли…
03. Николая Чудотворца — Ярославль | score=0.9577
    ↳ Николая Чудотворца. Город: Ярославль. храм в Ярославле Описание по фото: Кирпичное здание с зеленой крышей и башнями в парке | Церковь с зелеными куполами и крутыми башнями в травяной местности | Церковь с зелеными купол…
04. Гостиный

## 6) Генерация ответов: загрузка LLM

На этом этапе подключаем генеративную модель `Qwen2.5-7B-Instruct`.
Она используется в двух режимах:
- **baseline без RAG** — модель отвечает только по вопросу пользователя;
- **RAG-режим** — в prompt дополнительно подаётся контекст из retrieval.

Ниже загружаем токенизатор и саму модель в 4-bit режиме, чтобы запуск был возможен в Colab / Kaggle / GPU-среде с ограниченной памятью.

In [97]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,  # на T4 чаще всего ок
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.float16,   # вместо torch_dtype (у тебя предупреждение, что deprecated)
)

model.eval()
print("Loaded:", MODEL_ID)
print("Device map:", getattr(model, "hf_device_map", None))

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-7B-Instruct
Device map: None


In [98]:
import re

PHOTO_RE = re.compile(r"(?:\bОписание по фото\s*:\s*)(.*?)(?=\s*\|\s*|$)", flags=re.IGNORECASE | re.DOTALL)

def clean_retr_text(text: str) -> str:
    if text is None:
        return ""
    t = str(text)
    # убираем "Описание по фото: ...." (иногда их несколько)
    t = PHOTO_RE.sub("", t)
    # чистим мусорные двойные пробелы/разделители
    t = re.sub(r"\s+\|\s+", " | ", t)
    t = re.sub(r"\s{2,}", " ", t).strip(" | \n\t")
    return t.strip()

# быстрый sanity check на 3 строках
pois_final["_retr_clean_preview"] = pois_final["retr_text_ru"].astype(str).map(clean_retr_text)
pois_final[["Name","City","retr_text_ru","_retr_clean_preview"]].head(3)

,Name,City,retr_text_ru,_retr_clean_preview
0,1941-1945,Владимир,1941-1945. Город: Владимир. Описание по фото: ...,1941-1945. Город: Владимир. | Вид на город со ...
1,2000 летию Рождества Христова,Владимир,2000 летию Рождества Христова. Город: Владимир...,2000 летию Рождества Христова. Город: Владимир...
2,Бобёр,Владимир,Бобёр. Город: Владимир. Описание по фото: бобё...,Бобёр. Город: Владимир. | хоккейный вратарь в ...


 подготовка lookup по place_id и сбор контекста

In [99]:
# удобный быстрый lookup
pois_by_id = pois_final.set_index("place_id", drop=False)

def fetch_rows_by_place_ids(place_ids):
    rows = []
    for pid in place_ids:
        if pid in pois_by_id.index:
            rows.append(pois_by_id.loc[pid])
    if len(rows) == 0:
        return pd.DataFrame(columns=pois_final.columns)
    return pd.DataFrame(rows)

def build_context_from_hits(hits_df: pd.DataFrame, max_docs: int = 6, max_chars_per_doc: int = 900) -> str:
    """
    hits_df: DataFrame минимум с колонкой place_id (и желательно score)
    Собираем компактный контекст для LLM.
    """
    if hits_df is None or len(hits_df) == 0:
        return ""

    # берём top max_docs
    ids = hits_df["place_id"].astype(str).tolist()[:max_docs]
    rows = fetch_rows_by_place_ids(ids)

    blocks = []
    for _, r in rows.iterrows():
        name = str(r.get("Name",""))
        city = str(r.get("City",""))
        txt = clean_retr_text(r.get("retr_text_ru",""))
        if max_chars_per_doc and len(txt) > max_chars_per_doc:
            txt = txt[:max_chars_per_doc].rstrip() + "…"

        blocks.append(
            f"### {name} — {city}\n"
            f"{txt}\n"
        )

    return "\n".join(blocks).strip()

retrieval wrapper: query + city-filter + hybrid

Тут важно: фильтруем по городу после retrieval, чтобы “музей в Нижнем” не тянул вообще всё подряд по стран

In [100]:
def retrieve_hybrid(query: str, city: str | None = None, top_k: int = 7, alpha: float = 0.3) -> pd.DataFrame:
    """
    Возвращает hits_df с колонками: place_id, score
    Требует, чтобы у тебя уже была функция hybrid_search(q, top_k, alpha)
    """
    # 1) общий поиск
    hits = hybrid_search(query, top_k=top_k*5, alpha=alpha)  # берём пул побольше
    hits = hits.copy()
    hits["place_id"] = hits["place_id"].astype(str)

    # 2) city-filter (если задан)
    if city is not None:
        city_norm = norm_city(city)
        hits = hits.merge(pois_final[["place_id","City"]], on="place_id", how="left")
        hits["City"] = hits["City"].astype(str)
        hits = hits[hits["City"] == city_norm].drop(columns=["City"])

    # 3) дедуп и финальный top_k
    hits = hits.drop_duplicates(subset=["place_id"], keep="first")
    hits = hits.head(top_k).reset_index(drop=True)
    return hits

In [101]:
retrieve_hybrid("музей в Нижнем Новгороде", city = "Нижний Новгород", top_k=7, alpha=0.2)

,place_id,score
0,wd:Q65175093,1.000000
1,wd:Q4165827,0.955657
2,wd:Q2503505,0.935652
3,wd:Q4329133,0.909056
4,wd:Q4345262,0.903994
5,wd:Q4516884,0.903387
6,wd:Q2403914,0.898300


## 7) Baseline-генерация без финальной постобработки

Сначала делаем **простой baseline**, чтобы увидеть, как модель отвечает на запрос на основе retrieved context в самом прямом варианте.

Этот блок полезен как промежуточная точка: он показывает базовую связку `retrieve -> prompt -> generate`, но ещё не решает все проблемы продакшен-качества (обрезание ответа, chat-template leakage, постобработка хвостов). Ниже мы улучшим этот вариант и соберём финальный RAG-генератор.

In [102]:
def make_messages(query: str, city: str | None, context: str) -> list[dict]:
    city_part = f" Город: {city}." if city else ""
    system = (
        "Ты туристический ассистент. Отвечай по-русски.\n"
        "Используй ТОЛЬКО факты из блока CONTEXT. Если в контексте нет нужной информации — так и скажи.\n"
        "Дай ответ структурировано: 3-5 пунктов, коротко и по делу.\n"
        "Если пользователь спрашивает 'куда сходить/чем заняться' — предложи 3-5 мест/идей из контекста.\n"
    )
    user = (
        f"ВОПРОС:{city_part}\n{query}\n\n"
        f"CONTEXT:\n{context}\n"
    )
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]

@torch.inference_mode()
def generate_answer_baseline(query: str, city: str | None = None, top_k: int = 5, alpha: float = 0.3,
                             max_new_tokens: int = 350, temperature: float = 0.3) -> dict:
    hits = retrieve_hybrid(query, city=city, top_k=top_k, alpha=alpha)
    context = build_context_from_hits(hits, max_docs=min(6, top_k))

    messages = make_messages(query, city, context)

    # Qwen chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.9,
        repetition_penalty=1.05
    )

    text = tokenizer.decode(out[0], skip_special_tokens=True)

    # небольшая “вырезка” только ассистентской части: часто после шаблона идёт весь диалог
    # попробуем достать последнее сообщение ассистента грубо:
    answer = text.split("assistant")[-1].strip() if "assistant" in text else text

    return {
        "answer": answer,
        "hits": hits,
        "context": context
    }

# Алиас для совместимости со старыми вызовами ниже по ноутбуку
generate_answer = generate_answer_baseline


### Быстрый smoke test baseline-генерации

Проверяем, что минимальная цепочка работает end-to-end:
1. вызываем hybrid retrieval;
2. собираем контекст;
3. генерируем ответ моделью;
4. визуально смотрим на ответ и список top-hit документов.

Это ещё не финальный вариант генератора — дальше будет более стабильная версия для RAG.

In [103]:
res = generate_answer_baseline(
    "лучшие музеи в Нижнем Новгороде",
    city="Нижний Новгород",
    top_k=7,
    alpha=0.3
)

print(res["answer"])
print("\n--- TOP HITS ---")
display(res["hits"])

### Лучшие музеи в Нижнем Новгороде:

1. **Усадьба Добролюбовых** — единственный в мире музей литературного критика и писателя Николая Александровича Добролюбова, расположенный в Нижнем Новгороде.
2. **Домик Каширина** — музей в Нижнем Новгороде, представляющий собой красный деревянный дом с забором и деревом перед ним.
3. **Парк имени А.С. Пушкина** — хотя это больше парк, там также есть музейная экспозиция и различные достопримечательности, связанные с культурой и историей города.

--- TOP HITS ---


,place_id,score
0,wd:Q65175093,1.000000
1,wd:Q2503505,0.964003
2,wd:Q4165827,0.929775
3,wd:Q4329133,0.913067
4,wd:Q4345262,0.903373
5,wd:Q4516884,0.902210
6,wd:Q2403914,0.892468


## 8) Финальный RAG-генератор

В этом разделе код специально разделён на несколько логических частей:
1. **постобработка ответа** — убираем хвосты, служебные роли и мусор;
2. **подготовка контекста** — собираем top-k документы в prompt-friendly формат;
3. **chat generation wrapper** — единая функция общения с Qwen;
4. **внешние интерфейсы** — `generate_answer_rag(...)` и `generate_answer_no_rag(...)`.

Такой разнос по функциям удобен и для отладки, и для будущего рефакторинга в отдельный `src/generation.py`.

### 8.1) Почему понадобился отдельный финальный генератор

После базового smoke test стало видно, что у LLM есть несколько типичных проблем:
- ответы иногда обрываются раньше, чем хотелось бы;
- модель может продолжить диалог служебными ролями (`user:`, `assistant:`);
- на хвосте ответа иногда появляются артефакты chat template или мусорные символы.

Поэтому ниже собираем **финальную, более стабильную версию генератора**, в которой отдельно контролируем:
- подготовку контекста;
- stop tokens / terminators;
- postprocessing ответа;
- режимы `with RAG` и `without RAG` через единый интерфейс.

### 8.1) Постобработка ответа

Первая служебная часть финального генератора. Здесь описываем, как очищаем сырой вывод модели:
- обрезаем хвосты после `user:` / `assistant:`;
- убираем артефакты chat-template;
- режем мусорные продолжения ответа.

Такой слой особенно полезен для chat-моделей, которые иногда продолжают диалог в виде текста, хотя нам нужен только итоговый ответ.

In [104]:


def postprocess_answer(text: str) -> str:
    if not text:
        return text

    cut_text = text

    # 1. Обрезаем, если модель начала следующий ход диалога
    stop_patterns = [
        r"\n\s*user\s*:",
        r"\n\s*User\s*:",
        r"\n\s*assistant\s*:",
        r"\n\s*Assistant\s*:",
        r"<\|im_end\|>",
        r"<\|endoftext\|>",
    ]
    for pattern in stop_patterns:
        m = re.search(pattern, cut_text)
        if m:
            cut_text = cut_text[:m.start()].strip()

    # 2. Обрезаем по китайским символам, если они внезапно начались в хвосте
    m = re.search(r"[\u4e00-\u9fff]+", cut_text)
    if m:
        cut_text = cut_text[:m.start()].strip()

    # 3. Убираем мусорный хвост
    cut_text = re.sub(r"\]\)\);?\s*$", "", cut_text).strip()

    # 4. Если последняя строка явно оборвана — можно её отбросить
    lines = cut_text.splitlines()
    cleaned = []
    for line in lines:
        low = line.strip().lower()
        if low.startswith("user:") or low.startswith("assistant:"):
            break
        cleaned.append(line)

    cut_text = "\n".join(cleaned).strip()

    return cut_text

### 8.2) Подготовка context-building и retrieval wrappers

Во второй части финального генератора собираем всё, что относится к контексту:
- проверяем, что `pois_final` содержит нужные колонки;
- создаём быстрый lookup по `place_id`;
- форматируем top-k hit'ы в единый текстовый контекст;
- прячем детали retrieval за функцию `_get_hits_hybrid(...)`.

Это упрощает дальнейший код генерации: ниже функции уже работают не с сырыми таблицами, а с готовым контекстом.

In [105]:


# ЯЧЕЙКА 7 — Быстрый E2E smoke test (fixed)


# --- (опционально) очистка "Описание по фото"
# PHOTO_RE = re.compile(r"(?:\bОписание по фото\s*:\s*)(.*?)(?=\s*\|\s*|$)", flags=re.IGNORECASE | re.DOTALL)
# def clean_retr_text(text: str) -> str:
#     if text is None:
#         return ""
#     t = str(text)
#     t = PHOTO_RE.sub("", t)
#     t = re.sub(r"\s+\|\s+", " | ", t)
#     t = re.sub(r"\s{2,}", " ", t).strip(" | \n\t")
#     return t.strip()
#
# # pois_final["_retr_clean"] = pois_final["retr_text_ru"].astype(str).map(clean_retr_text)

def _ensure_pois_index(df: pd.DataFrame) -> pd.DataFrame:
    if "place_id" not in df.columns:
        raise ValueError("pois_final должен содержать колонку 'place_id'")
    if "retr_text_ru" not in df.columns:
        raise ValueError("pois_final должен содержать колонку 'retr_text_ru'")
    return df

pois_final = _ensure_pois_index(pois_final)
pois_by_id = pois_final.set_index("place_id", drop=False)

def _format_context(hits_df: pd.DataFrame, max_docs: int = 5, max_chars_per_doc: int = 900) -> str:
    """
    Делает контекст для промпта из top-k результатов.
    hits_df ожидается с place_id и (опц) score.
    """
    chunks = []
    if hits_df is None or len(hits_df) == 0:
        return ""

    for i, r in hits_df.head(max_docs).iterrows():
        pid = str(r["place_id"])
        if pid not in pois_by_id.index:
            continue
        row = pois_by_id.loc[pid]
        name = str(row.get("Name", "")).strip()
        city = str(row.get("City", "")).strip()
        text = str(row.get("retr_text_ru", "")).strip()

        # если включишь чистку, замени строчку выше на:
        # text = str(row.get("_retr_clean", row.get("retr_text_ru", ""))).strip()

        if len(text) > max_chars_per_doc:
            text = text[:max_chars_per_doc] + "…"

        score = r["score"] if "score" in hits_df.columns else None
        score_str = f"{float(score):.4f}" if isinstance(score, (int, float)) else ""
        header = f"[{i+1}] {name} ({city}) {score_str}".strip()
        chunks.append(header + "\n" + text)

    return "\n\n---\n\n".join(chunks)

def _get_hits_hybrid(query: str, city: str | None, top_k: int = 7, alpha: float = 0.3) -> pd.DataFrame:
    """
    Пытаемся использовать твою функцию retrieve_hybrid.
    Если её нет — fallback на hybrid_search + city filter.
    """
    if "retrieve_hybrid" in globals():
        # ожидаем что retrieve_hybrid вернёт DataFrame (часто уже с Name/City/retr_text_ru)
        hits = retrieve_hybrid(query, city=city, top_k=top_k, alpha=alpha)
        # нормализуем формат к ["place_id","score"]
        if isinstance(hits, pd.DataFrame):
            cols = hits.columns.tolist()
            if "place_id" in cols:
                out = hits.copy()
                if "score" not in out.columns:
                    # если скор назывался иначе
                    for c in ["hybrid_score", "final_score"]:
                        if c in out.columns:
                            out = out.rename(columns={c: "score"})
                            break
                    if "score" not in out.columns:
                        out["score"] = None
                return out[["place_id","score"]]
        raise ValueError("retrieve_hybrid() вернул неожиданный формат")
    else:
        # fallback
        if "hybrid_search" not in globals():
            raise ValueError("Ни retrieve_hybrid, ни hybrid_search не определены")
        hits = hybrid_search(query, top_k=top_k, alpha=alpha)
        if city is not None and "City" in pois_final.columns:
            hits = hits.merge(pois_final[["place_id","City"]], on="place_id", how="left")
            hits = hits[hits["City"] == city].drop(columns=["City"])
        return hits

### 8.3) Финальные функции генерации

Здесь определяем общий chat wrapper и два пользовательских интерфейса:
- `generate_answer_rag(...)` — основной вариант для проекта;
- `generate_answer_no_rag(...)` — baseline без контекста, чтобы сравнить, что даёт retrieval.

Именно `generate_answer_rag(...)` дальше используется в e2e-оценке и в финальных примерах.

In [106]:
def _generate_chat(
    model,
    tokenizer,
    messages,
    max_new_tokens=420,
    min_new_tokens=80,
    temperature=0.0,
    repetition_penalty=1.08
    ):
    device = model.device

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # для chat-моделей Qwen часто полезно остановиться
    # либо по eos, либо по специальному chat terminator
    terminators = [tokenizer.eos_token_id]
    try:
        assistant_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
        if assistant_end_id is not None and assistant_end_id != tokenizer.unk_token_id:
            terminators.append(assistant_end_id)
    except Exception:
        pass

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            do_sample=False,
            repetition_penalty=repetition_penalty,
            eos_token_id=terminators,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    prompt_len = inputs["input_ids"].shape[1]
    gen_ids = outputs[0][prompt_len:]
    answer = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    return postprocess_answer(answer)

def generate_answer_rag(
    query: str,
    city: str | None = None,
    top_k: int = 7,
    alpha: float = 0.3,
    max_docs: int = 5
):
    hits = _get_hits_hybrid(query, city=city, top_k=top_k, alpha=alpha)
    context = _format_context(hits, max_docs=max_docs)

    system = (
    "Ты русскоязычный туристический ассистент.\n"
    "Отвечай только на русском языке.\n"
    "Никогда не используй китайский, английский или любой другой язык.\n"
    "Даже если во входе или контексте встречаются фрагменты на других языках, ответ всё равно должен быть полностью на русском.\n"
    "Используй только факты из предоставленного контекста.\n"
    "Если данных в контексте недостаточно, честно скажи об этом.\n"
    "Не выдумывай факты.\n"
    "Сформируй законченный, полный ответ.\n"
    "Не обрывай ответ на полуслове.\n"
    "Если перечисляешь места, доводи каждую рекомендацию до конца."
    )

    user = (
    f"Вопрос пользователя: {query}\n\n"
    f"Город (если задан): {city}\n\n"
    f"Контекст из базы:\n{context}\n\n"
    "Сформируй ответ только на русском языке.\n"
    "Если уместно, предложи 5 вариантов мест.\n"
    "Для каждого варианта дай 1-2 полных предложения пояснения.\n"
    "Ответ должен быть завершённым и не обрываться.\n"
    "Не добавляй информацию, которой нет в контексте."
    )

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]

    answer = _generate_chat(
    model,
    tokenizer,
    messages,
    max_new_tokens=420,
    min_new_tokens=120,
    temperature=0.0,
    # top_p=1.0,
    repetition_penalty=1.08
    )
    return {"answer": answer, "hits": hits}

def generate_answer_no_rag(query: str, city: str | None = None):
    system = (
        "Ты туристический ассистент. Отвечай ТОЛЬКО на русском.\n"
        "Если не уверен — говори, что это общие рекомендации."
    )
    user = f"Вопрос: {query}\nГород (если задан): {city}\nОтветь кратко и по делу."
    messages = [{"role":"system","content":system},{"role":"user","content":user}]
    answer = _generate_chat(model, tokenizer, messages)
    return {"answer": answer}

# ------------------------
# SMOKE TEST (RAG vs NO-RAG)
# ------------------------
test_cases = [
    {"query": "лучшие музеи в Нижнем Новгороде", "city": "Нижний Новгород"},
    {"query": "куда сходить в Ярославле с детьми", "city": "Ярославль"},
    {"query": "что посмотреть в Екатеринбурге за 1 день", "city": "Екатеринбург"},
]

for ex in test_cases:
    q, c = ex["query"], ex["city"]
    print("="*120)
    print(f"QUERY: {q} | CITY: {c}\n")

    rag = generate_answer_rag(q, city=c, top_k=7, alpha=0.3, max_docs=5)
    print("RAG ANSWER:\n", rag["answer"], "\n")
    display(rag["hits"].head(7))

    base = generate_answer_no_rag(q, city=c)
    print("\nNO-RAG ANSWER:\n", base["answer"])

QUERY: лучшие музеи в Нижнем Новгороде | CITY: Нижний Новгород

RAG ANSWER:
 Вот пять лучших музеев в Нижнем Новгороде:

1. Усадьба Добролюбовых - это уникальный музей в Нижнем Новгороде, посвященный литературному критику и писателю Николаю Добролюбову. Здесь можно узнать больше о его жизни и творчестве.

2. Домик Каширина - музей в Нижнем Новгороде, который рассказывает о жизни и традициях русского села XIX века. Это интересное место для тех, кто хочет познакомиться с культурой прошлого.

3. Музей А.С. Пушкина в Нижнем Новгороде - хотя он не указан в контексте как отдельный музей, этот парк культуры и отдыха связан с Пушкиным и содержит множество экспонатов, связанных с его жизнью и творчеством.

4. Музей истории Нижнего Новгорода - хотя информации о нем нет в данном контексте, он также является одним из лучших музеев города, где можно узнать о истории и культуре региона.

5. Музей железнодорожной эпохи - хотя информация о нем тоже отсутствует, он предлагает посетителям познакомиться 

,place_id,score
0,wd:Q65175093,1.000000
1,wd:Q2503505,0.964003
2,wd:Q4165827,0.929775
3,wd:Q4329133,0.913067
4,wd:Q4345262,0.903373
5,wd:Q4516884,0.902210
6,wd:Q2403914,0.892468



NO-RAG ANSWER:
 Лучшие музеи в Нижнем Новгороде:

1. Музей изобразительных искусств им. С.С. Уварова
2. Музей истории города Нижнего Новгорода
3. Музей-заповедник "Архангельский"
4. Музей природы имени А.П. Ковалевского
5. Музей космонавтики имени Ю.А. Гагарина

Эти музеи предлагают разнообразные экспозиции, отражающие историю, культуру и природу Нижнего Новгорода.
QUERY: куда сходить в Ярославле с детьми | CITY: Ярославль

RAG ANSWER:
 Вот пять вариантов мест для посещения с детьми в Ярославле:

1. Ярославский государственный театр юного зрителя им. В. С. Розова - здесь можно посмотреть интересные спектакли для детей и взрослых, а также посетить выставки и мастер-классы.

2. Часовня Александра Невского - это красивое архитектурное сооружение с интересными деталями и историей, которая может заинтересовать детей.

3. Успенский собор - величественная церковь с золотыми куполами, где можно показать детям красоту русской архитектуры и познакомиться с историей города.

4. Парк имени Ф. Г. 

,place_id,score
0,wd:Q4538871,1.000000
1,wd:Q4343395,0.980371
2,wd:Q4507592,0.976158
3,wd:Q1140551,0.974135
4,wd:Q4504845,0.971940
5,wd:Q30913942,0.960168
6,wd:Q16715656,0.954807



NO-RAG ANSWER:
 В Ярославле с детьми можно посетить:

1. Ярославский кремль с музеями и памятниками архитектуры.
2. Музей-заповедник "Ярославль cổский".
3. Парк имени Ленина с детской игровой площадкой и прудом.
4. Детскую творческую мастерскую "Квартал".
5. Зоопарк "Лебяжий остров".

Эти местаCombine the given pieces of information to create a coherent response.

These places offer a mix of history, culture, and entertainment suitable for families with children in Ярославле.
QUERY: что посмотреть в Екатеринбурге за 1 день | CITY: Екатеринбург

RAG ANSWER:
 Вот 5 мест, которые стоит посетить в Екатеринбурге за один день:

1. Площадь 1-й Пятилетки - это центральная площадь города с интересными архитектурными элементами и историческими зданиями. Здесь можно прогуляться и ознакомиться с архитектурными достопримечательностями.

2. Динамо - современный спортивный комплекс с красивыми зданиями и парком. Можно пройтись по парку, осмотреть здание комплекса и даже воспользоваться услугами одно

,place_id,score
0,wd:Q4365466,0.923337
1,wd:Q37996725,0.880327
2,wd:Q55210808,0.865382
3,wd:Q4411412,0.857930
4,wd:Q50359307,0.809855
5,wd:Q4306077,0.809332
6,wd:Q2036269,0.797925



NO-RAG ANSWER:
 За один день в Екатеринбурге можно посетить:

1. Уральский музей истории геологии, минERALогии и нефти (Магнитный сквер).
2. Архитектурный ансамбль Свято-Троицкого собора.
3. Краеведческий музей (ул. Ленина, 80).
4. Площадь Революции с памятником Петру I.
5. Мемориальный комплекс "Политехнический" и памятник Юрию Гагарину.
6. Экспофорум (выставочный центр).
7. Дворец Республики (посмотреть фасад).

Эти места покажут основные достопримечательности города.


## 9) Автоматическая оценка генерации

LLM-as-a-Judge здесь пока не используем: для ноутбука важнее сначала получить воспроизводимый локальный baseline. Поэтому ниже считаем две понятные proxy-метрики:
- **Answer Relevancy** — отвечает ли модель на исходный вопрос;
- **Faithfulness** — насколько ответ опирается на retrieved context.

Эти метрики не идеальны, но хорошо подходят для сравнения версий пайплайна и первичной диагностики качества.

Перед блоком метрик важно помнить:
- высокое `answer_relevancy` ещё не гарантирует groundedness;
- низкое `faithfulness` не всегда означает "плохой ответ" — иногда метрика просто слишком строгая к перефразу;
- поэтому результаты ниже стоит читать как **диагностический сигнал**, а не как абсолютную истину.

### 9.1) Answer Relevancy

Ниже — **локальная приближённая версия** метрики `Answer Relevancy`, близкая по идее к RAGAS.

Что измеряем:
- насколько ответ реально отвечает на исходный вопрос;
- не ушёл ли ответ в сторону;
- не стал ли слишком общим / нерелевантным.

Идея:
1. из готового **ответа** генерируем несколько обратных вопросов;
2. считаем их embedding similarity с исходным вопросом;
3. берём среднее сходство.

Это не официальный RAGAS-скор, но он хорошо подходит для:
- локальной оценки без внешних API;
- сравнения версий пайплайна;
- демонстрации понимания метрики в pet-project.


In [107]:

def _split_lines_to_questions(text: str) -> List[str]:
    lines = []
    for raw in text.splitlines():
        s = raw.strip()
        if not s:
            continue
        s = re.sub(r"^[-•\d\.)\s]+", "", s).strip()
        if s:
            lines.append(s)
    return lines

def _generate_questions_from_answer(answer: str, k: int = 3) -> List[str]:
    """Генерируем k вопросов, для которых данный answer выглядел бы хорошим ответом."""
    prompt = (
    f"Сгенерируй {k} разных вопросов, на которые данный текст является хорошим ответом.\n"
    "Пиши только на русском языке.\n"
    "Никогда не используй китайский, английский или любой другой язык.\n"
    "Каждый вопрос пиши с новой строки.\n"
    "Без нумерации.\n"
    "Без пояснений.\n"
    "Без вступительного текста.\n"
    "Выведи только сами вопросы.\n\n"
    f"ТЕКСТ:\n{answer}\n\n"
    "ВОПРОСЫ:"
    )
    out = generate_answer_no_rag(prompt, city=None)["answer"]
    qs = _split_lines_to_questions(out)

    if len(qs) == 0:
        qs = [answer[:120]]
    while len(qs) < k:
        qs.append(qs[-1])
    return qs[:k]

def answer_relevancy(question: str, answer: str, k: int = CONFIG.ar_num_questions) -> Dict[str, Any]:
    """Возвращает score и диагностическую информацию."""
    generated_questions = _generate_questions_from_answer(answer, k=k)

    q_emb = embedder.encode([question], normalize_embeddings=True)
    gen_emb = embedder.encode(generated_questions, normalize_embeddings=True)

    sims = (q_emb @ gen_emb.T).flatten()
    return {
        "answer_relevancy": float(np.mean(sims)),
        "generated_questions": generated_questions,
        "similarities": [float(x) for x in sims]
    }

# sanity check
_ar_demo = answer_relevancy(
    "Что посмотреть вечером в городе, если люблю атмосферные прогулки?",
    "Можно выбрать исторический центр, красивую набережную и архитектурные места для спокойной прогулки."
)
_ar_demo


{'answer_relevancy': 0.8552761673927307,
 'generated_questions': ['Где можно пройтись по историческому центру?',
  'Какие места лучше выбрать для пеших прогулок?',
  'Какие достопримечательности стоит посетить для спокойной прогулки?<tool_call>'],
 'similarities': [0.8341505527496338, 0.8551899790763855, 0.8764879703521729]}

### 9.2) Faithfulness / Groundedness

`Faithfulness` отвечает уже не за соответствие вопросу, а за **опору ответа на retrieved context**.

В этом ноутбуке используем **упрощённую proxy-версию**:
1. разбиваем ответ на claims;
2. для каждого claim ищем наиболее похожее предложение в контексте;
3. считаем token-overlap;
4. если overlap выше порога, считаем claim поддержанным.

Важно: это не полноценная LLM-based factual verification, а лёгкая локальная эвристика. Она полезна для сравнения версий пайплайна, но может занижать score при хорошем перефразе.

In [108]:
from typing import List, Tuple

# Вспомогательные функции

def simple_tokenize(text: str) -> List[str]:
    if text is None:
        return []
    text = str(text).lower().replace("ё", "е")
    return re.findall(r"[a-zа-я0-9]+", text, flags=re.IGNORECASE)


def split_into_sentences(text: str) -> List[str]:
    """
    Более аккуратное разбиение на предложения/claim-ы.
    Также умеет работать со списками и markdown-строками.
    """
    if not text:
        return []

    text = str(text)

    # убираем грубые markdown-артефакты
    text = text.replace("**", " ")
    text = text.replace("*", " ")
    text = text.replace("•", "\n")
    text = text.replace("—", " - ")
    text = text.replace("–", " - ")

    # нумерованные списки переводим в переносы строк
    text = re.sub(r"\n?\s*\d+[\.\)]\s+", "\n", text)

    # режем и по предложениям, и по строкам
    parts = re.split(r"[\.!?]\s+|\n+", text)

    parts = [p.strip() for p in parts if p and p.strip()]
    return parts


def clean_claim_text(text: str) -> str:
    text = str(text)

    # markdown / нумерация / ссылки / мусор
    text = re.sub(r"\*\*", " ", text)
    text = re.sub(r"`+", " ", text)
    text = re.sub(r"\[.*?\]\(.*?\)", " ", text)
    text = re.sub(r"^\s*\d+[\.\)]\s*", "", text)
    text = re.sub(r"^\s*[-–—]\s*", "", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def is_bad_claim(text: str, min_chars: int = 12, min_tokens: int = 3) -> bool:
    """
    Отбрасываем мусор вроде:
    - 'С'
    - '2'
    - слишком короткие обрывки
    """
    if not text:
        return True

    text = text.strip()

    if len(text) < min_chars:
        return True

    toks = simple_tokenize(text)

    if len(toks) < min_tokens:
        return True

    if re.fullmatch(r"[\d\W_]+", text):
        return True

    return False


def extract_claims(answer: str, max_claims: int = 12) -> List[str]:
    parts = split_into_sentences(answer)

    claims = []
    for p in parts:
        p = clean_claim_text(p)
        if is_bad_claim(p):
            continue
        claims.append(p)

    return claims[:max_claims]


def extract_context_sentences(contexts: List[str]) -> List[str]:
    sents = []
    for c in contexts:
        parts = split_into_sentences(c)
        for p in parts:
            p = clean_claim_text(p)
            if is_bad_claim(p, min_chars=8, min_tokens=2):
                continue
            sents.append(p)
    return sents


def jaccard_overlap(a: str, b: str) -> float:
    a_toks = set(simple_tokenize(a))
    b_toks = set(simple_tokenize(b))

    if len(a_toks) == 0 or len(b_toks) == 0:
        return 0.0

    inter = len(a_toks & b_toks)
    union = len(a_toks | b_toks) + 1e-9
    return inter / union


# ----------------------------------------
# Основная метрика faithfulness
# ----------------------------------------

def faithfulness(
    answer: str,
    contexts: List[str],
    min_overlap: float = 0.18,
    max_claims: int = 12
) -> Tuple[float, pd.DataFrame]:
    """
    Proxy-faithfulness:
    - разбиваем ответ на claim-ы
    - для каждого claim ищем лучшее предложение из context
    - считаем token overlap (Jaccard)
    - если overlap >= min_overlap -> supported=True

    Возвращает:
    - итоговый score
    - DataFrame с деталями
    """

    claims = extract_claims(answer, max_claims=max_claims)
    ctx_sents = extract_context_sentences(contexts)

    rows = []

    if len(claims) == 0:
        details = pd.DataFrame(columns=["claim", "best_ctx_sentence", "overlap", "supported"])
        return 0.0, details

    if len(ctx_sents) == 0:
        details = pd.DataFrame(
            [{"claim": cl, "best_ctx_sentence": "", "overlap": 0.0, "supported": False} for cl in claims]
        )
        return 0.0, details

    supported_count = 0

    for cl in claims:
        best_sent = ""
        best_score = 0.0

        for sent in ctx_sents:
            score = jaccard_overlap(cl, sent)
            if score > best_score:
                best_score = score
                best_sent = sent

        is_supported = best_score >= min_overlap
        supported_count += int(is_supported)

        rows.append({
            "claim": cl,
            "best_ctx_sentence": best_sent,
            "overlap": round(float(best_score), 4),
            "supported": bool(is_supported)
        })

    details = pd.DataFrame(rows)
    score = supported_count / max(1, len(rows))

    return float(score), details

## 10) End-to-end оценка RAG

Теперь объединяем retrieval, generation и метрики в единый eval loop.

Что делаем:
- задаём небольшой набор тестовых запросов;
- запускаем `generate_answer_rag(...)`;
- считаем `answer_relevancy`;
- считаем `faithfulness`;
- сохраняем артефакты (`hits`, `contexts`, подробности метрик) для ручного разбора.

Этот блок нужен уже не для разработки отдельных функций, а для **целостной оценки поведения пайплайна**.

In [109]:
import pandas as pd
from typing import List, Dict, Any

# Восстановление contexts из hits

def hits_to_contexts(
    hits_df: pd.DataFrame,
    max_docs: int = 5,
    max_chars_per_doc: int = 900
) -> List[str]:
    contexts = []

    if hits_df is None or len(hits_df) == 0:
        return contexts

    for _, r in hits_df.head(max_docs).iterrows():
        pid = str(r["place_id"])
        if pid not in pois_by_id.index:
            continue

        row = pois_by_id.loc[pid]
        name = str(row.get("Name", "")).strip()
        city = str(row.get("City", "")).strip()
        text = str(row.get("retr_text_ru", "")).strip()

        if max_chars_per_doc and len(text) > max_chars_per_doc:
            text = text[:max_chars_per_doc].rstrip() + "…"

        block = f"{name} — {city}. {text}".strip()
        contexts.append(block)

    return contexts


# ----------------------------------------
# Один запрос end-to-end
# ----------------------------------------

def run_one_e2e(
    question: str,
    city: str | None = None,
    top_k: int = 7,
    alpha: float = 0.3,
    max_docs: int = 5,
    ar_k: int | None = None,
    faith_min_overlap: float = 0.18
) -> Dict[str, Any]:
    if ar_k is None:
        ar_k = CONFIG.ar_num_questions

    out = generate_answer_rag(
        query=question,
        city=city,
        top_k=top_k,
        alpha=alpha,
        max_docs=max_docs
    )

    answer = out["answer"]
    hits = out["hits"]
    contexts = hits_to_contexts(hits, max_docs=max_docs)

    ar_obj = answer_relevancy(question, answer, k=ar_k)
    faith_score, faith_df = faithfulness(
        answer=answer,
        contexts=contexts,
        min_overlap=faith_min_overlap,
        max_claims=12
    )

    return {
        "question": question,
        "city": city,
        "answer": answer,
        "hits": hits,
        "contexts": contexts,
        "answer_relevancy": float(ar_obj["answer_relevancy"]),
        "answer_relevancy_details": ar_obj,
        "faithfulness": float(faith_score),
        "faithfulness_details": faith_df
    }


# ----------------------------------------
# Батчевый прогон
# ----------------------------------------

def run_e2e(
    queries,
    top_k: int = 7,
    alpha: float = 0.3,
    max_docs: int = 5,
    ar_k: int | None = None,
    faith_min_overlap: float = 0.18
):
    rows = []
    raw_outputs = []

    for item in queries:
        if isinstance(item, dict):
            question = item.get("question", "")
            city = item.get("city", None)
        else:
            question = str(item)
            city = None

        out = run_one_e2e(
            question=question,
            city=city,
            top_k=top_k,
            alpha=alpha,
            max_docs=max_docs,
            ar_k=ar_k,
            faith_min_overlap=faith_min_overlap
        )

        raw_outputs.append(out)

        rows.append({
            "question": out["question"],
            "city": out["city"],
            "answer_relevancy": out["answer_relevancy"],
            "faithfulness": out["faithfulness"],
            "n_contexts": len(out["contexts"]),
            "answer_preview": out["answer"][:250].replace("\n", " ")
        })

    df = pd.DataFrame(rows)

    if len(df) > 0:
        print("=== Средние метрики ===")
        print(f"Answer Relevancy: {df['answer_relevancy'].mean():.4f}")
        print(f"Faithfulness:     {df['faithfulness'].mean():.4f}")

    return df, raw_outputs

In [110]:
test_queries = [
    {"question": "Лучшие музеи в Нижнем Новгороде", "city": "Нижний Новгород"},
    {"question": "Куда сходить в Ярославле с детьми?", "city": "Ярославль"},
    {"question": "Посоветуй красивые места для вечерней прогулки", "city": None},
    {"question": "Есть ли места с природой и видами?", "city": None},
    {"question": "Что посмотреть любителю истории и архитектуры?", "city": None},
]

df_e2e, raw_e2e = run_e2e(
    test_queries,
    top_k=7,
    alpha=0.3,
    max_docs=5,
    ar_k=CONFIG.ar_num_questions,
    faith_min_overlap=0.18
)

display(df_e2e)

=== Средние метрики ===
Answer Relevancy: 0.8922
Faithfulness:     0.3013


,question,city,answer_relevancy,faithfulness,n_contexts,answer_preview
0,Лучшие музеи в Нижнем Новгороде,Нижний Новгород,0.885682,0.363636,5,Вот пять лучших музеев в Нижнем Новгороде: 1....
1,Куда сходить в Ярославле с детьми?,Ярославль,0.967445,0.142857,5,Вот пять вариантов мест для посещения с детьми...
2,Посоветуй красивые места для вечерней прогулки,NaN,0.869636,0.166667,5,Вот пять красивых мест для вечерней прогулки: ...
3,Есть ли места с природой и видами?,NaN,0.839021,0.000000,5,В данном контексте нет прямых указаний на мест...
4,Что посмотреть любителю истории и архитектуры?,NaN,0.898983,0.833333,5,Вот 5 вариантов мест для любителей истории и а...


### Как интерпретировать результаты метрик

- **Answer Relevancy** отвечает на вопрос: *попадает ли ответ в запрос пользователя?*
- **Faithfulness** отвечает на вопрос: *можно ли подтвердить утверждения ответа retrieved context-ом?*

Возможна ситуация, когда ответ выглядит полезным и связным, но `faithfulness` остаётся низким: это означает, что модель отвечает по теме, однако часть формулировок додумывает или слишком свободно перефразирует относительно retrieved context.

## 11) Выводы

Что получилось в проекте:
- собран и очищен POI-level датасет по достопримечательностям;
- реализованы BM25, dense и hybrid retrieval;
- собран RAG-пайплайн на базе `Qwen2.5-7B-Instruct`;
- добавлены локальные метрики `answer_relevancy` и `faithfulness`;
- проведён end-to-end прогон на наборе тестовых запросов.

Главный практический вывод: **hybrid retrieval + контекст в prompt заметно улучшают содержательность ответов**, но groundedness всё ещё остаётся местом для улучшения. В будущем сюда планируется добавить:
- более строгий LLM-as-a-Judge;
- дополнительную настройку prompt / reranking;
